# V2 Feature Merge

## 목적

01~05b에서 각각 만든 V2 Feature와 V1 최종 데이터(`df_final_v1`)를 합쳐
모델 3개가 쓸 데이터셋을 만든다.

| 산출물 | 단위 | 쓰는 모델 |
|---|---|---|
| `v2_trade.parquet` | 거래 (234,302) | 11 회귀 (거래가격) |
| `v2_region_month.parquet` | 구 × 예측시점 월 (1,500) | 31 분류 (향후 3개월 +3% 상승) |
| `v2_complex.parquet` | 단지 (7,871) | 21 군집 + 웹 단지 카드 |
| `v2_feature_registry.csv` | 열 | 열마다 출처·역할·시점 규칙 (11/21/31이 Feature를 고르는 기준) |

## 시점 규칙 (미래정보 누출 방지)

구×월 행 m은 **m월 1일에 알 수 있던 상태**다. 거래 행은 계약월의 이 상태 + 계약일 기준 Feature를 쓴다.

| 자료 | 공개 시점 | 사용 |
|---|---|---|
| 아파트 매매·전월세 실거래 | 계약 후 30일 안에 신고 | m-2월까지 (**확정월**: m월 1일에 신고 기한이 지난 가장 최근 월) |
| 한국부동산원 월간 통계 (01) | 익월 15일 | m-2월 (01에서 수정) |
| 한국은행 CSI·CCSI (01) | 당월 하순 | m-1월 |
| 한국은행 대출 가중평균금리 | 익월 말 | m-2월 |
| 기준금리 | 결정 즉시 | m-1월 |
| 정책·공급·인구 (03·04·05) | 각 노트북의 공표 규칙 | 이미 반영 |
| 역·역 출입구·대규모점포 | 개통일·인허가/폐업일 | 계약일(역·출입구)·계약월 1일(점포)에 운영 중인 시설만 |
| 주요 일자리 (직장인구 상위 10개 행정동) | 분기 종료 2개월 뒤 | 분기 시작 + 5개월부터, 분기마다 다시 선정 |
| 유치원 | 개원일 | 개원일 ≤ 계약일 (현재 운영 목록이라 폐원 기관 없음) |
| 응급의료기관 | 현재 지정 목록만 | V1 값 유지, 레지스트리 `ablation` (11에서 포함/제외 비교) |

## V1에서 바뀌는 것

- V1 `구_전월_*`·`서울_전월_*`·전세/월세·대출금리 피처는 m-1월 값을 써서 월 초·중순 거래에 아직 신고·공표되지 않은 자료가 들어갔다.
  → 같은 계산을 **확정월(m-2) 기준**으로 다시 만들고, V1 값은 `_v1`을 붙여 11에서 비교용으로만 남긴다.
- V1 역세권거리·대규모점포거리·유치원거리는 현재 시설 목록 기준 → 거래 시점에 운영 중이던 시설로 다시 계산 (V1 값은 `_v1`).
- V1 주요일자리거리는 2021Q1~2026Q1 전체 분기 평균으로 고른 상위 10개 행정동 기준(미래 분기 포함) → 분기별 선정 + 공표 시차로 다시 계산.
- 역 출입구 좌표로 `역출입구거리_m`을 새로 만든다 (21 교통 태그).
- 위치 ID처럼 쓰인 이름형 범주(`최근접*` 이름·분류)와 V1 정책 5개 열(03으로 대체)은 뺀다.
- 2개월 시차로 앞쪽 결측이 늘지 않도록 **2020-01~2021-06 매매 실거래(워밍업)** 를 시장 통계 계산에만 쓴다 (`06a_collect_trade_warmup.py`). 학습 행은 그대로 2021-07부터다.

## 검증 (모두 assert)

1. 재현: 시차 1로 다시 계산하면 V1 값과 같아야 한다 (구·서울 시장, 전세·월세, 금리, 역 거리, 분류 정답).
2. 잘라내기: 행 m에 대해 m-1월 이후 자료를 지우고 다시 계산해도 값이 같아야 한다. 시차 1이면 이 검사가 실패해야 한다(시험의 시험).
3. 결합: 행 수 유지, 1:1 결합, 열 이름 충돌 없음.
4. 금지 열(정답·거래 이후 정보)이 모델 Feature에 없어야 한다.
5. 결측·학습구간 상수 Feature 보고.

In [1]:
# ============================================================
# 경로·공통 설정
# ============================================================
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
from pyproj import Transformer
from sklearn.neighbors import BallTree

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 60)


def find_project_root(start=Path.cwd()):
    for path in [start, *start.parents]:
        if (path / "data").exists():
            return path
    raise FileNotFoundError("프로젝트 루트에서 data 폴더를 찾을 수 없습니다.")


PROJECT_DIR = find_project_root()
EXTERNAL_DIR = PROJECT_DIR / "data" / "external"
PROCESSED_DIR = PROJECT_DIR / "data" / "v2_5year_seoul" / "processed"

FIRST_MONTH, LAST_MONTH = pd.Timestamp("2021-07-01"), pd.Timestamp("2026-06-01")
MONTHS = pd.date_range(FIRST_MONTH, LAST_MONTH, freq="MS")
TRAIN_END, VALID_END = pd.Timestamp("2025-07-01"), pd.Timestamp("2026-01-01")
TRADE_LAG = 2      # 실거래 30일 신고 → m월 1일에는 m-2월까지 확정
EARTH_RADIUS_M = 6_371_000


def split_of(dates):
    return np.select([dates < TRAIN_END, dates < VALID_END], ["train", "valid"], "test")


v1 = pd.read_parquet(PROCESSED_DIR / "df_final_v1.parquet")
v1["거래월일자"] = pd.to_datetime(v1["거래월일자"])
v1["시군구명"] = v1["시군구명"].astype(str)
v1["법정동"] = v1["법정동"].astype(str)
v1["단지일련번호"] = v1["단지일련번호"].astype(str)
GU_NAME = v1.drop_duplicates("법정동시군구코드").set_index("법정동시군구코드")["시군구명"]
print("V1:", v1.shape, v1["거래월일자"].min().date(), "~", v1["거래월일자"].max().date())
print("구:", v1["시군구명"].nunique(), "| 단지:", v1["단지일련번호"].nunique())

V1: (234302, 124) 2021-07-01 ~ 2026-06-01
구: 25 | 단지: 7871


## 1. 시장 통계용 매매 거래 풀 (워밍업 + V1)

- V1 정제본(2021-07~)에 워밍업 실거래(2020-01~2021-06)를 붙인다. 워밍업은 V1과 같게 해제 거래(`해제여부 = O`)만 뺀다.
- V1은 해제 거래 외에 좌표 변환 실패 거래도 뺐다. 그 차이가 워밍업과 V1 사이 표본 차이이므로 크기를 확인한다.

In [2]:
# ============================================================
# 거래 풀
# ============================================================
POOL_COLS = ["법정동시군구코드", "시군구명", "법정동", "단지일련번호", "거래월일자", "거래금액", "평당가"]

warm = pd.read_parquet(PROCESSED_DIR / "apt_trade_warmup_raw.parquet")
warm = warm[warm["해제여부"].ne("O")].copy()
warm["법정동시군구코드"] = warm["법정동시군구코드"].astype(str).str.zfill(5)
warm["시군구명"] = warm["법정동시군구코드"].map(GU_NAME)
warm["법정동"] = warm["법정동"].astype(str).str.strip()
warm["단지일련번호"] = warm["단지일련번호"].astype(str)
warm["거래월일자"] = pd.to_datetime(dict(year=warm["계약년도"], month=warm["계약월"], day=1))
warm["거래금액"] = warm["거래금액"].astype(float)
warm["평당가"] = warm["거래금액"] / (warm["전용면적"].astype(float) / 3.3058)
assert warm["시군구명"].notna().all() and (warm["거래월일자"] < FIRST_MONTH).all()

pool_v1 = v1[POOL_COLS].copy()
pool_v1["거래금액"] = pool_v1["거래금액"].astype(float)
pool = pd.concat([warm[POOL_COLS], pool_v1], ignore_index=True)

# 정제 기준 차이: V1은 해제 거래 + 좌표 실패 거래를 뺐다 → 2021-07 이후 원본에서 해제만 뺀 건수와 비교
raw_after = pd.read_parquet(PROCESSED_DIR / "apt_trade_raw.parquet", columns=["해제여부"])
not_cancelled = int(raw_after["해제여부"].ne("O").sum())
print(f"워밍업 거래: {len(warm):,}  ({warm['거래월일자'].min().date()} ~ {warm['거래월일자'].max().date()})")
print(f"2021-07 이후 해제 제외 원본 {not_cancelled:,} vs V1 {len(v1):,} → 좌표 실패로 빠진 비율 {1 - len(v1) / not_cancelled:.2%}")
print("구×월 거래가 없는 칸:", 25 * pool["거래월일자"].nunique() - pool.groupby(["시군구명", "거래월일자"]).ngroups)

워밍업 거래: 106,696  (2020-01-01 ~ 2021-06-01)
2021-07 이후 해제 제외 원본 234,302 vs V1 234,302 → 좌표 실패로 빠진 비율 0.00%
구×월 거래가 없는 칸: 0


## 2. 구·서울 매매 시장 피처: V1 재현 → 확정월(m-2) 재계산

V1 정의 그대로(구·서울 월 중위가격·중위평당가·거래량, 직전 3·6개월 월중위값의 중위, 거래량 합, 1개월 변화율)
시차만 바꾼다. 이름은 `전월` → `확정월`, `과거N개월` → `확정N개월`.

In [3]:
# ============================================================
# 구·서울 시장 피처
# ============================================================
def lag_label(name, lag):
    if lag == 1:
        return name
    return name.replace("전월", "확정월").replace("과거3개월", "확정3개월").replace("과거6개월", "확정6개월")


def monthly_market(trades):
    gu = (trades.groupby(["시군구명", "거래월일자"])
          .agg(월중위거래금액=("거래금액", "median"), 월중위평당가=("평당가", "median"), 월거래량=("거래금액", "size"))
          .reset_index())
    months = pd.date_range(trades["거래월일자"].min(), trades["거래월일자"].max(), freq="MS")
    full = pd.MultiIndex.from_product([sorted(gu["시군구명"].unique()), months], names=["시군구명", "거래월일자"])
    gu = gu.set_index(["시군구명", "거래월일자"]).reindex(full).reset_index()
    gu["월거래량"] = gu["월거래량"].fillna(0)
    seoul = (trades.groupby("거래월일자")
             .agg(서울_월중위가격=("거래금액", "median"), 서울_월거래량=("거래금액", "size"))
             .reindex(months).rename_axis("거래월일자").reset_index())
    seoul["서울_월거래량"] = seoul["서울_월거래량"].fillna(0)
    return gu, seoul


def roll(s, lag, window, how):
    r = s.shift(lag).rolling(window, min_periods=window)
    return r.median() if how == "median" else r.sum()


def market_features(trades, lag):
    gu, seoul = monthly_market(trades)
    out = gu[["시군구명", "거래월일자"]].copy()
    g = gu.groupby("시군구명")
    L = lambda n: lag_label(n, lag)
    out[L("구_전월_중위가격")] = g["월중위거래금액"].shift(lag)
    out[L("구_전월_중위평당가")] = g["월중위평당가"].shift(lag)
    out[L("구_전월_거래량")] = g["월거래량"].shift(lag)
    out[L("구_과거3개월_중위가격")] = g["월중위거래금액"].transform(lambda s: roll(s, lag, 3, "median"))
    out[L("구_과거6개월_중위가격")] = g["월중위거래금액"].transform(lambda s: roll(s, lag, 6, "median"))
    out[L("구_과거3개월_거래량")] = g["월거래량"].transform(lambda s: roll(s, lag, 3, "sum"))
    out[L("구_과거6개월_거래량")] = g["월거래량"].transform(lambda s: roll(s, lag, 6, "sum"))
    out[L("구_전월_가격변화율")] = g["월중위거래금액"].transform(lambda s: s.shift(lag) / s.shift(lag + 1) - 1)

    s = seoul[["거래월일자"]].copy()
    p, n = seoul["서울_월중위가격"], seoul["서울_월거래량"]
    s[L("서울_전월_중위가격")] = p.shift(lag)
    s[L("서울_과거3개월_중위가격")] = roll(p, lag, 3, "median")
    s[L("서울_과거6개월_중위가격")] = roll(p, lag, 6, "median")
    s[L("서울_전월_거래량")] = n.shift(lag)
    s[L("서울_과거3개월_거래량")] = roll(n, lag, 3, "sum")
    s[L("서울_전월_가격변화율")] = p.shift(lag) / p.shift(lag + 1) - 1
    return out.merge(s, on="거래월일자", how="left")


V1_MARKET_COLS = ["구_전월_중위가격", "구_전월_중위평당가", "구_전월_거래량", "구_과거3개월_중위가격",
                  "구_과거6개월_중위가격", "구_과거3개월_거래량", "구_과거6개월_거래량", "구_전월_가격변화율",
                  "서울_전월_중위가격", "서울_과거3개월_중위가격", "서울_과거6개월_중위가격",
                  "서울_전월_거래량", "서울_과거3개월_거래량", "서울_전월_가격변화율"]
MARKET_COLS = [lag_label(c, TRADE_LAG) for c in V1_MARKET_COLS]

# 재현: V1 거래만, 시차 1 → V1과 같아야 한다
rep = v1[["transaction_id", "시군구명", "거래월일자"]].merge(market_features(pool_v1, 1), on=["시군구명", "거래월일자"], how="left")
diff = {c: float(np.nanmax(np.abs(rep[c].astype(float).to_numpy() - v1[c].astype(float).to_numpy()))) for c in V1_MARKET_COLS}
nan_mismatch = {c: int((rep[c].isna() != v1[c].isna()).sum()) for c in V1_MARKET_COLS}
print("V1 재현 최대 절대오차:", max(diff.values()), "| 결측 위치 불일치:", sum(nan_mismatch.values()))
assert max(diff.values()) < 1e-6 and sum(nan_mismatch.values()) == 0

market_gu = market_features(pool, TRADE_LAG)
market_gu = market_gu[market_gu["거래월일자"].between(FIRST_MONTH, LAST_MONTH)].reset_index(drop=True)
print("확정월 기준 구×월 시장 피처:", market_gu.shape)
display(market_gu[MARKET_COLS].isna().sum().to_frame("결측 (1,500행 중)").T)

V1 재현 최대 절대오차: 0.0 | 결측 위치 불일치: 0
확정월 기준 구×월 시장 피처: (1500, 16)


,구_확정월_중위가격,구_확정월_중위평당가,구_확정월_거래량,구_확정3개월_중위가격,구_확정6개월_중위가격,구_확정3개월_거래량,구_확정6개월_거래량,구_확정월_가격변화율,서울_확정월_중위가격,서울_확정3개월_중위가격,서울_확정6개월_중위가격,서울_확정월_거래량,서울_확정3개월_거래량,서울_확정월_가격변화율
"결측 (1,500행 중)",0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [4]:
# ============================================================
# 잘라내기 검증: 행 m은 m-2월까지의 거래만으로 계산돼야 한다
# - m-1·m월 거래는 구마다 가짜 1건(1원)으로 바꾼다 (행을 만들기 위한 자리표시). 값이 쓰이면 결과가 달라진다.
# ============================================================
def placeholder(month_list):
    # 행을 만들기 위한 자리표시 거래: 구마다 1건, 가격 1원 → 피처 계산에 쓰이면 값이 달라진다
    return (pool[pool["거래월일자"].isin(month_list)].drop_duplicates(["시군구명", "거래월일자"])
            .assign(거래금액=1.0, 평당가=1.0))


def truncation_bad(lag):
    cols = [lag_label(c, lag) for c in V1_MARKET_COLS]
    full = market_features(pool, lag).set_index(["시군구명", "거래월일자"])
    bad = 0
    for m in MONTHS:
        known = pool[pool["거래월일자"] <= m - pd.DateOffset(months=TRADE_LAG)]
        fake = placeholder([m - pd.DateOffset(months=1), m])
        part = market_features(pd.concat([known, fake]), lag).set_index(["시군구명", "거래월일자"])
        a = full.xs(m, level="거래월일자")[cols]
        b = part.xs(m, level="거래월일자")[cols].reindex(a.index)
        bad += int((~np.isclose(a.to_numpy(float), b.to_numpy(float), equal_nan=True)).any(axis=1).sum())
    return bad


bad = truncation_bad(TRADE_LAG)
print("m-1·m월 거래를 가짜 값으로 바꿨을 때 달라진 구×월:", bad)
assert bad == 0
bad_leaky = truncation_bad(1)
print("시험의 시험 — V1 시차(1)로 계산하면 잡히는 구×월:", bad_leaky)
assert bad_leaky > 0

m-1·m월 거래를 가짜 값으로 바꿨을 때 달라진 구×월: 0


시험의 시험 — V1 시차(1)로 계산하면 잡히는 구×월: 1500


## 3. 전세·월세 피처: V1 재현 → 확정월 재계산

V1 `05_rent` 정의를 그대로 옮긴다 (전세 = 월세 0, 전세평당가 = 보증금 / 평, 단지 → 법정동 → 자치구 fallback,
단지는 최근 3·6개월 3건 이상일 때만, 전세가율 = 전세평당가 / 같은 범위 매매평당가).
전월세도 계약 후 30일 신고라 시차를 2로 바꾸고, 매매평당가는 워밍업을 포함한 거래 풀로 계산한다.

재계산에서 바꾼 점 하나 더 (잘라내기 검증에서 발견):
V1은 동·단지 패널을 전체 기간으로 만들어, **나중에 처음 거래가 생기는 동**에도 과거 '0건' 행이 생겼다.
그러면 자치구 값으로 넘어가야 할 자리에 0이 들어가 '이 동에 앞으로 거래가 생긴다'는 미래정보가 섞인다.
재계산 버전은 그룹이 처음 관측된 달부터만 패널을 만들고, 월세 피처는 전세 패널에 바깥 결합한다.

In [5]:
# ============================================================
# 전월세 원본 전처리 (V1과 동일)
# ============================================================
ROLLING_WINDOWS = (3, 6)
PRICE_MIN_MONTHS = {3: 2, 6: 3}
APT_MIN_TRADES = 3

rent = pd.read_parquet(PROCESSED_DIR / "apt_rent_raw.parquet").replace(["nan", "None", "null", ""], pd.NA)
for col in ["dealYear", "dealMonth", "dealDay", "deposit", "monthlyRent", "excluUseAr"]:
    rent[col] = pd.to_numeric(rent[col].astype("string").str.replace(",", "", regex=False), errors="coerce")
for col in ["aptNm", "umdNm"]:
    rent[col] = rent[col].astype("string").str.strip()
rent["sggCd"] = rent["sggCd"].astype("string").str.split(".").str[0].str.zfill(5)
rent["aptSeq"] = rent["aptSeq"].astype("string").str.strip()
rent["거래일자"] = pd.to_datetime({"year": rent["dealYear"], "month": rent["dealMonth"], "day": rent["dealDay"]}, errors="coerce")
rent["거래월일자"] = rent["거래일자"].dt.to_period("M").dt.to_timestamp()
rent["계약유형"] = pd.Series(pd.NA, index=rent.index, dtype="string")
rent.loc[rent["monthlyRent"].eq(0), "계약유형"] = "전세"
rent.loc[rent["monthlyRent"].gt(0), "계약유형"] = "월세"
jeonse = rent[rent["계약유형"].eq("전세") & rent["deposit"].gt(0) & rent["excluUseAr"].gt(0) & rent["거래월일자"].notna()].copy()
jeonse["전세평당가"] = jeonse["deposit"] / (jeonse["excluUseAr"] / 3.3058)
print("전월세:", len(rent), "| 전세:", len(jeonse), "|", rent["거래월일자"].min().date(), "~", rent["거래월일자"].max().date())

전월세: 1415810 | 전세: 805108 | 2021-01-01 ~ 2026-06-01


In [6]:
# ============================================================
# 전세·월세 Lag 피처 (V1 함수에 시차 인자만 추가)
# ============================================================
def complete_panel(monthly, group_cols, count_cols, months, observed_start=False):
    groups = monthly[group_cols].drop_duplicates()
    panel = (groups.merge(pd.DataFrame({"거래월일자": months}), how="cross")
             .merge(monthly, on=group_cols + ["거래월일자"], how="left", validate="one_to_one"))
    if observed_start:
        # 그룹(동·단지)이 처음 관측된 달부터만 행을 만든다.
        # V1처럼 전체 기간 패널을 만들면 '나중에 처음 거래가 생기는 동'에도 과거 0건 행이 생겨
        # 자치구 fallback 대신 0이 들어간다 → 미래에 거래가 생긴다는 정보가 섞인다.
        first = monthly.groupby(group_cols)["거래월일자"].min().rename("_first").reset_index()
        panel = panel.merge(first, on=group_cols, how="left")
        panel = panel[panel["거래월일자"] >= panel["_first"]].drop(columns="_first")
    for col in count_cols:
        panel[col] = panel[col].fillna(0).astype("int64")
    return panel.sort_values(group_cols + ["거래월일자"]).reset_index(drop=True)


def grouped_roll(panel, group_cols, col, lag, window, min_periods, how):
    shifted = panel.groupby(group_cols, sort=False)[col].shift(lag)
    keys = [panel[c] for c in group_cols]
    r = shifted.groupby(keys, sort=False).rolling(window, min_periods=min_periods)
    r = r.median() if how == "median" else r.sum()
    return r.reset_index(level=list(range(len(group_cols))), drop=True).reindex(panel.index)


def lag_features(panel, group_cols, prefix, name, price_col, volume_col, lag, labels):
    out = panel.copy()
    g = out.groupby(group_cols, sort=False)
    out[f"{prefix}_{labels[0]}_{name}평당가"] = g[price_col].shift(lag)
    out[f"{prefix}_{labels[0]}_{name}거래량"] = g[volume_col].shift(lag)
    for w, label in zip(ROLLING_WINDOWS, labels[1:]):
        out[f"{prefix}_{label}_{name}평당가"] = grouped_roll(out, group_cols, price_col, lag, w, PRICE_MIN_MONTHS[w], "median")
        out[f"{prefix}_{label}_{name}거래량"] = grouped_roll(out, group_cols, volume_col, lag, w, w, "sum")
    return out


def mix_features(rent_df, group_cols, prefix, lag, labels, months, observed_start):
    src = rent_df[rent_df["계약유형"].isin(["전세", "월세"]) & rent_df["거래월일자"].notna()].copy()
    src["월세건"] = src["계약유형"].eq("월세").astype(int)
    monthly = src.groupby(group_cols + ["거래월일자"]).agg(월월세=("월세건", "sum"), 월전체=("월세건", "size")).reset_index()
    panel = complete_panel(monthly, group_cols, ["월월세", "월전체"], months, observed_start)
    g = panel.groupby(group_cols, sort=False)
    out = panel[group_cols + ["거래월일자"]].copy()
    prev_m, prev_t = g["월월세"].shift(lag), g["월전체"].shift(lag)
    out[f"{prefix}_{labels[0]}_월세거래량"] = prev_m
    out[f"{prefix}_{labels[0]}_월세비중"] = prev_m / prev_t.replace(0, np.nan)
    for w, label in zip(ROLLING_WINDOWS, labels[1:]):
        m = grouped_roll(panel, group_cols, "월월세", lag, w, w, "sum")
        t = grouped_roll(panel, group_cols, "월전체", lag, w, w, "sum")
        out[f"{prefix}_{label}_월세거래량"] = m
        out[f"{prefix}_{label}_월세비중"] = m / t.replace(0, np.nan)
    return out


def rent_features(rent_df, jeonse_df, sale_trades, targets, lag):
    labels = ["전월", "과거3개월", "과거6개월"] if lag == 1 else ["확정월", "확정3개월", "확정6개월"]
    v1_mode = lag == 1   # 시차 1 = V1 재현 (V1의 전체 기간 패널·왼쪽 결합 그대로)
    observed = not v1_mode
    rent_months = pd.date_range(rent_df["거래월일자"].min(), rent_df["거래월일자"].max(), freq="MS")
    sale_trades = sale_trades.astype({"법정동시군구코드": "string", "법정동": "string"})
    # V1은 매매 패널도 전월세 기간으로 만들었다. 재계산은 매매 풀(워밍업 포함) 기간을 쓴다.
    sale_months = rent_months if lag == 1 else pd.date_range(sale_trades["거래월일자"].min(), rent_df["거래월일자"].max(), freq="MS")

    def jeonse_level(keys, prefix):
        monthly = jeonse_df.groupby(keys + ["거래월일자"]).agg(월중위전세평당가=("전세평당가", "median"),
                                                          월전세거래량=("전세평당가", "size")).reset_index()
        panel = complete_panel(monthly, keys, ["월전세거래량"], rent_months, observed)
        return lag_features(panel, keys, prefix, "전세", "월중위전세평당가", "월전세거래량", lag, labels)

    def sale_level(keys, trade_keys, prefix):
        monthly = (sale_trades.groupby(trade_keys + ["거래월일자"]).agg(월중위매매평당가=("평당가", "median"),
                                                                      월매매거래량=("평당가", "size"))
                   .reset_index().rename(columns=dict(zip(trade_keys, keys))))
        panel = complete_panel(monthly, keys, ["월매매거래량"], sale_months, observed)
        return lag_features(panel, keys, prefix, "매매", "월중위매매평당가", "월매매거래량", lag, labels)

    gu = jeonse_level(["sggCd"], "자치구")
    dong = jeonse_level(["sggCd", "umdNm"], "법정동")
    apt = jeonse_level(["aptSeq", "sggCd", "umdNm"], "단지")
    # 재계산은 바깥 결합: 전세 없이 월세만 있는 동도 월세 피처를 가진다 (V1은 전세 패널에 왼쪽 결합)
    mix_how = "left" if v1_mode else "outer"
    gu = gu.merge(mix_features(rent_df, ["sggCd"], "자치구", lag, labels, rent_months, observed),
                  on=["sggCd", "거래월일자"], how=mix_how)
    dong = dong.merge(mix_features(rent_df, ["sggCd", "umdNm"], "법정동", lag, labels, rent_months, observed),
                      on=["sggCd", "umdNm", "거래월일자"], how=mix_how)
    sale_gu = sale_level(["sggCd"], ["법정동시군구코드"], "자치구")
    sale_dong = sale_level(["sggCd", "umdNm"], ["법정동시군구코드", "법정동"], "법정동")
    gu = gu.merge(sale_gu[["sggCd", "거래월일자"] + [f"자치구_{l}_매매평당가" for l in labels]], on=["sggCd", "거래월일자"], how="left")
    dong = dong.merge(sale_dong[["sggCd", "umdNm", "거래월일자"] + [f"법정동_{l}_매매평당가" for l in labels]],
                      on=["sggCd", "umdNm", "거래월일자"], how="left")
    for l in labels:
        gu[f"자치구_{l}_전세가율"] = gu[f"자치구_{l}_전세평당가"] / gu[f"자치구_{l}_매매평당가"].replace(0, np.nan)
        dong[f"법정동_{l}_전세가율"] = dong[f"법정동_{l}_전세평당가"] / dong[f"법정동_{l}_매매평당가"].replace(0, np.nan)

    t = targets[["transaction_id", "법정동시군구코드", "법정동", "단지일련번호", "거래월일자"]].copy()
    t["법정동시군구코드"] = t["법정동시군구코드"].astype(str).str.zfill(5).astype("string")
    t["법정동"] = t["법정동"].astype("string")
    t["단지일련번호"] = t["단지일련번호"].astype("string")
    keep = lambda d, p: [c for c in d.columns if c.startswith(p + "_")]
    t = t.merge(gu[["sggCd", "거래월일자"] + keep(gu, "자치구")].rename(columns={"sggCd": "법정동시군구코드"}),
                on=["법정동시군구코드", "거래월일자"], how="left", validate="many_to_one")
    t = t.merge(dong[["sggCd", "umdNm", "거래월일자"] + keep(dong, "법정동")]
                .rename(columns={"sggCd": "법정동시군구코드", "umdNm": "법정동"}),
                on=["법정동시군구코드", "법정동", "거래월일자"], how="left", validate="many_to_one")
    t = t.merge(apt[["aptSeq", "sggCd", "umdNm", "거래월일자"] + keep(apt, "단지")]
                .rename(columns={"aptSeq": "단지일련번호", "sggCd": "법정동시군구코드", "umdNm": "법정동"}),
                on=["단지일련번호", "법정동시군구코드", "법정동", "거래월일자"], how="left", validate="many_to_one")
    assert len(t) == len(targets)

    out = t[["transaction_id"]].copy()
    for label, min_trades in zip(labels, [1, APT_MIN_TRADES, APT_MIN_TRADES]):
        apt_vol = t[f"단지_{label}_전세거래량"]
        apt_price = t[f"단지_{label}_전세평당가"].where(apt_vol >= min_trades)
        dong_price, gu_price = t[f"법정동_{label}_전세평당가"], t[f"자치구_{label}_전세평당가"]
        out[f"전세평당가_{label}"] = apt_price.combine_first(dong_price).combine_first(gu_price)
        out[f"전세거래량_{label}"] = (apt_vol.where(apt_price.notna()).combine_first(t[f"법정동_{label}_전세거래량"])
                                     .combine_first(t[f"자치구_{label}_전세거래량"]))
        out[f"전세가율_{label}"] = t[f"법정동_{label}_전세가율"].combine_first(t[f"자치구_{label}_전세가율"])
        out[f"전세피처출처_{label}"] = np.select(
            [apt_price.notna(), apt_price.isna() & dong_price.notna(), apt_price.isna() & dong_price.isna() & gu_price.notna()],
            ["단지", "법정동", "자치구"], default="결측")
        out[f"월세거래량_{label}"] = t[f"법정동_{label}_월세거래량"].combine_first(t[f"자치구_{label}_월세거래량"])
        out[f"월세비중_{label}"] = t[f"법정동_{label}_월세비중"].combine_first(t[f"자치구_{label}_월세비중"])
    return out, gu


V1_RENT_COLS = [f"{n}_{l}" for l in ["전월", "과거3개월", "과거6개월"]
                for n in ["전세평당가", "전세거래량", "전세가율", "전세피처출처", "월세거래량", "월세비중"]]
RENT_COLS = [lag_label(c, TRADE_LAG) for c in V1_RENT_COLS]

sale_v1 = v1[["법정동시군구코드", "법정동", "거래월일자", "평당가"]].assign(법정동시군구코드=lambda d: d["법정동시군구코드"].astype(str).str.zfill(5))
rep_rent, _ = rent_features(rent, jeonse, sale_v1, v1, 1)
rep_rent = rep_rent.set_index("transaction_id").loc[v1["transaction_id"]]
worst = 0.0
for c in V1_RENT_COLS:
    a, b = rep_rent[c].to_numpy(), v1[c].to_numpy()
    if c.startswith("전세피처출처"):
        assert (a == b.astype(str)).all(), c
    else:
        a, b = a.astype(float), b.astype(float)
        assert (np.isnan(a) == np.isnan(b)).all(), c
        worst = max(worst, float(np.nanmax(np.abs(a - b) / np.maximum(np.abs(b), 1))))
print("V1 전세·월세 피처 재현 최대 상대오차:", worst)
assert worst < 1e-9

sale_pool = pool[["법정동시군구코드", "법정동", "거래월일자", "평당가"]]
rent_trade, rent_gu = rent_features(rent, jeonse, sale_pool, v1, TRADE_LAG)
print("확정월 기준 전세·월세 피처:", rent_trade.shape)
display(rent_trade[RENT_COLS].isna().mean().round(4).to_frame("결측률").T)

V1 전세·월세 피처 재현 최대 상대오차: 0.0


확정월 기준 전세·월세 피처: (234302, 19)


,전세평당가_확정월,전세거래량_확정월,전세가율_확정월,전세피처출처_확정월,월세거래량_확정월,월세비중_확정월,전세평당가_확정3개월,전세거래량_확정3개월,전세가율_확정3개월,전세피처출처_확정3개월,월세거래량_확정3개월,월세비중_확정3개월,전세평당가_확정6개월,전세거래량_확정6개월,전세가율_확정6개월,전세피처출처_확정6개월,월세거래량_확정6개월,월세비중_확정6개월
결측률,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.02,0.0,0.0,0.02,0.02


In [7]:
# ============================================================
# 잘라내기 검증 (표본 6개월): m-1월 이후 전월세·매매 자료를 지우고 m월 거래의 피처를 다시 계산
# ============================================================
CHECK_MONTHS = pd.to_datetime(["2021-09-01", "2022-03-01", "2023-06-01", "2024-12-01", "2025-08-01", "2026-05-01"])


def rent_truncation_bad(lag, full, months):
    labels_cols = [lag_label(c, lag) for c in V1_RENT_COLS]
    bad = 0
    for m in months:
        cut = m - pd.DateOffset(months=TRADE_LAG)
        targets = v1[v1["거래월일자"].eq(m)]
        # 패널이 m월까지 이어지도록 m월 행은 남기되, m-1월 이후 '거래 내용'은 쓰지 않았는지 보려고 m-1월 자료를 지운다
        r = rent[(rent["거래월일자"] <= cut) | (rent["거래월일자"] == m)]
        j = jeonse[(jeonse["거래월일자"] <= cut) | (jeonse["거래월일자"] == m)]
        s = (sale_pool if lag != 1 else sale_v1)
        s = s[(s["거래월일자"] <= cut) | (s["거래월일자"] == m)]
        part, _ = rent_features(r, j, s, targets, lag)
        a = full.set_index("transaction_id").loc[targets["transaction_id"], labels_cols]
        b = part.set_index("transaction_id").loc[targets["transaction_id"], labels_cols]
        for c in labels_cols:
            if c.startswith("전세피처출처"):
                bad += int((a[c].to_numpy() != b[c].to_numpy()).sum())
            else:
                bad += int((~np.isclose(a[c].to_numpy(float), b[c].to_numpy(float), equal_nan=True)).sum())
    return bad


bad = rent_truncation_bad(TRADE_LAG, rent_trade, CHECK_MONTHS)
print("m-1월 자료를 지웠을 때 달라진 값:", bad)
assert bad == 0
rep_full = rep_rent.reset_index()
bad_leaky = rent_truncation_bad(1, rep_full, CHECK_MONTHS[:2])
print("시험의 시험 — V1 시차(1)에서 잡히는 값:", bad_leaky)
assert bad_leaky > 0

m-1월 자료를 지웠을 때 달라진 값: 0


시험의 시험 — V1 시차(1)에서 잡히는 값: 63302


## 4. 금리: 대출금리만 공표 시차 반영

기준금리는 결정 즉시 공개되므로 V1(m-1) 그대로 쓴다.
주담대·전세대출 가중평균금리(한국은행, 익월 말 공표)는 m-2월 값으로 바꾼다. 이름은 `_최근공표`.

In [8]:
# ============================================================
# 금리 피처
# ============================================================
base_raw = pd.read_csv(EXTERNAL_DIR / "한국은행_기준금리.csv").melt(var_name="연월", value_name="기준금리")
base_raw = base_raw[base_raw["연월"].astype(str).str.match(r"\d{4}[./-]\d{2}")].copy()
base_raw["거래월일자"] = pd.to_datetime(base_raw["연월"].astype(str).str.replace(".", "-", regex=False)
                                     .str.replace("/", "-", regex=False) + "-01", errors="coerce")
base_raw["기준금리"] = pd.to_numeric(base_raw["기준금리"], errors="coerce")
loan_raw = pd.read_csv(EXTERNAL_DIR / "mortgage_rent_loan_rate_monthly.csv")
month_cols = [c for c in loan_raw.columns if str(c)[:4].isdigit()]
loan = loan_raw[["계정항목"] + month_cols].melt(id_vars="계정항목", var_name="연월", value_name="금리")
loan["거래월일자"] = pd.to_datetime(loan["연월"].str.extract(r"(\d{4}/\d{2})")[0] + "/01", format="%Y/%m/%d")
loan["금리"] = pd.to_numeric(loan["금리"], errors="coerce")
loan = loan.pivot(index="거래월일자", columns="계정항목", values="금리").reset_index()
loan.columns = [str(c).strip() for c in loan.columns]
loan = loan.rename(columns={"주택담보대출": "주담대금리", "전세자금대출": "전세대출금리"})
rate_monthly = (base_raw[["거래월일자", "기준금리"]].dropna(subset=["거래월일자"])
                .merge(loan, on="거래월일자", how="inner").sort_values("거래월일자").reset_index(drop=True))
rate_monthly = rate_monthly[rate_monthly["거래월일자"] >= "2020-12-01"].reset_index(drop=True)
print("금리 원자료:", rate_monthly["거래월일자"].min().date(), "~", rate_monthly["거래월일자"].max().date())

RATE_LAG = {"기준금리": 1, "주담대금리": 2, "전세대출금리": 2}


def rate_features(lags, frame=None):
    frame = rate_monthly if frame is None else frame
    out = frame[["거래월일자"]].copy()
    for col, lag in lags.items():
        s = frame[col]
        level = f"{col}_전월" if lag == 1 else f"{col}_최근공표"
        suffix = "" if lag == 1 or col == "기준금리" else "_최근공표"
        out[level] = s.shift(lag)
        out[f"{col}_3개월변화{suffix}"] = s.shift(lag) - s.shift(lag + 3)
        out[f"{col}_6개월변화{suffix}"] = s.shift(lag) - s.shift(lag + 6)
    return out


V1_RATE_COLS = [f"{c}_{s}" for c in ["기준금리", "주담대금리", "전세대출금리"] for s in ["전월", "3개월변화", "6개월변화"]]
rep_rate = v1[["거래월일자"]].merge(rate_features({c: 1 for c in RATE_LAG}), on="거래월일자", how="left")
rate_diff = max(float(np.nanmax(np.abs(rep_rate[c].to_numpy(float) - v1[c].to_numpy(float)))) for c in V1_RATE_COLS)
print("V1 금리 피처 재현 최대 절대오차:", rate_diff)
assert rate_diff < 1e-9

rates = rate_features(RATE_LAG)
rates = rates[rates["거래월일자"].between(FIRST_MONTH, LAST_MONTH)].reset_index(drop=True)
RATE_COLS = [c for c in rates.columns if c != "거래월일자"]
NEW_RATE_COLS = [c for c in RATE_COLS if not c.startswith("기준금리")]
print("금리 피처:", RATE_COLS)
display(rates[RATE_COLS].isna().sum().to_frame("결측 월 수").T)

# 잘라내기: m월 행은 대출금리 m-2월, 기준금리 m-1월까지만 써야 한다
bad = 0
for m in MONTHS:
    known = rate_monthly.copy()
    for col, lag in RATE_LAG.items():
        known.loc[known["거래월일자"] > m - pd.DateOffset(months=lag), col] = np.nan
    part = rate_features(RATE_LAG, known).set_index("거래월일자").loc[m, RATE_COLS]
    full_row = rates.set_index("거래월일자").loc[m, RATE_COLS]
    bad += int((~np.isclose(full_row.to_numpy(float), part.to_numpy(float), equal_nan=True)).sum())
print("공표 전 금리를 지웠을 때 달라진 값:", bad)
assert bad == 0

금리 원자료: 2020-12-01 ~ 2026-07-01
V1 금리 피처 재현 최대 절대오차: 0.0
금리 피처: ['기준금리_전월', '기준금리_3개월변화', '기준금리_6개월변화', '주담대금리_최근공표', '주담대금리_3개월변화_최근공표', '주담대금리_6개월변화_최근공표', '전세대출금리_최근공표', '전세대출금리_3개월변화_최근공표', '전세대출금리_6개월변화_최근공표']


,기준금리_전월,기준금리_3개월변화,기준금리_6개월변화,주담대금리_최근공표,주담대금리_3개월변화_최근공표,주담대금리_6개월변화_최근공표,전세대출금리_최근공표,전세대출금리_3개월변화_최근공표,전세대출금리_6개월변화_최근공표
결측 월 수,0,0,0,0,0,1,0,0,1


공표 전 금리를 지웠을 때 달라진 값: 0


## 5. 위치: 거래 시점에 운영 중이던 역·대규모점포

- **역:** V1 역 목록(BallTree haversine)에서 기간 중 개통한 역은 개통일 이후 거래에만 쓴다.
  (신림선 11개역 2022-05-28, 남위례 2021-12-18, 8호선 별내선 연장 2024-08-10, 4호선 진접선 2022-03-19)
- **대규모점포:** 인허가·폐업·휴업·취소 일자로 계약월 1일에 영업 중이던 점포만 쓴다.
  좌표 없는 점포는 `06b_geocode_large_stores.py`로 보완했다. V1 수동 제외 2건(주소 오류 통일상가, 공항시장)은 그대로 뺀다.
- **역 출입구:** 서울교통빅데이터플랫폼 출입구 좌표(`06c_download_subway_entrances.py`)로 가장 가까운 출입구까지 거리 `역출입구거리_m`을 새로 만든다.
  출입구는 최근접 역의 개통일을 이어받고, 출입구 자료가 없는 역은 역 좌표를 쓴다. 역 대표 좌표보다 실제로 걸어서 닿는 지점에 가깝다 (21 교통 태그 "역 출입구 250m/500m 이내"에 사용, 공식 역세권 판정 아님).
- **주요 일자리:** V1은 2021Q1~2026Q1 **전체 분기 평균**으로 직장인구 상위 10개 행정동을 골랐다 (미래 분기 포함).
  분기마다 상위 10개를 다시 고르고, 분기 자료는 분기 시작 + 5개월부터 쓴다
  (서울시 골목상권 제공정보: 분기 종료 2개월 뒤 갱신, 예: 1분기 → 5월 말 → 6월 1일부터).
- **유치원:** 개원일 이후 거래에만 그 유치원을 쓴다. 원천이 현재 운영 목록이라 그 사이 폐원한 유치원은 없다 (한계).
- **응급의료기관:** 원천 API가 현재 지정 목록만 주고 과거 목록이 없어 V1 값을 그대로 쓴다 (한계, 11에서 Ablation으로 제외 여부 확인).

In [9]:
# ============================================================
# 역: 개통일 반영
# ============================================================
def nearest_m(src_lat, src_lon, dst_lat, dst_lon):
    tree = BallTree(np.radians(np.column_stack([dst_lat, dst_lon])), metric="haversine")
    d, _ = tree.query(np.radians(np.column_stack([src_lat, src_lon])), k=1)
    return d[:, 0] * EARTH_RADIUS_M


stations = pd.read_excel(EXTERNAL_DIR / "subway_stations.xlsx")
OPENED = {  # (노선명 포함 문자열, 역사명) → 개통일
    ("신림선", None): "2022-05-28",
    ("8호선", "남위례"): "2021-12-18",
    ("8호선", "암사역사공원"): "2024-08-10", ("8호선", "장자호수공원"): "2024-08-10",
    ("8호선", "구리(구리전통시장)"): "2024-08-10", ("8호선", "동구릉"): "2024-08-10",
    ("광역철도 8호선", "다산역"): "2024-08-10", ("광역철도 8호선", "별내역"): "2024-08-10",
    ("진접선", None): "2022-03-19",
}
stations["개통일"] = pd.NaT
for (line, name), day in OPENED.items():
    mask = stations["노선명"].astype(str).str.contains(line) & (stations["역사명"].eq(name) if name else True)
    assert mask.any(), (line, name)
    stations.loc[mask, "개통일"] = pd.Timestamp(day)
print("기간 중 개통 역:", stations["개통일"].notna().sum())

# 재현: 전체 역으로 계산하면 V1 역세권거리와 같아야 한다
coords = v1[["위도", "경도"]].drop_duplicates().reset_index(drop=True)
coords["전체역"] = nearest_m(coords["위도"], coords["경도"], stations["역위도"], stations["역경도"])
chk = v1[["위도", "경도", "역세권거리_m"]].merge(coords, on=["위도", "경도"], how="left")
print("V1 역세권거리 재현 최대 오차(m):", float((chk["전체역"] - chk["역세권거리_m"]).abs().max()))
assert (chk["전체역"] - chk["역세권거리_m"]).abs().max() < 1e-6

# 개통일 사이 구간마다 운영 역만으로 계산
cuts = sorted(stations["개통일"].dropna().unique())
trade_station = v1[["transaction_id", "위도", "경도", "계약일자"]].copy()
trade_station["역세권거리_m"] = np.nan
for start, end in zip([pd.Timestamp.min] + list(cuts), list(cuts) + [pd.Timestamp.max]):
    active = stations[stations["개통일"].isna() | (stations["개통일"] <= start)]
    rows = trade_station["계약일자"].ge(start) & trade_station["계약일자"].lt(end)
    if rows.any():
        trade_station.loc[rows, "역세권거리_m"] = nearest_m(trade_station.loc[rows, "위도"], trade_station.loc[rows, "경도"],
                                                       active["역위도"], active["역경도"])
changed = (trade_station["역세권거리_m"] - v1["역세권거리_m"]).abs() > 1e-6
print(f"개통 전 거래로 거리가 바뀐 거래: {changed.sum():,}건 / 단지 {v1.loc[changed, '단지일련번호'].nunique()}곳")
display((trade_station.loc[changed, "역세권거리_m"] - v1.loc[changed, "역세권거리_m"]).describe().round(0).to_frame("거리 증가(m)").T)
# 마지막 개통 이후 거래는 V1과 같아야 한다
assert not changed[v1["계약일자"] >= cuts[-1]].any()

기간 중 개통 역: 21
V1 역세권거리 재현 최대 오차(m): 0.0


개통 전 거래로 거리가 바뀐 거래: 960건 / 단지 103곳


,count,mean,std,min,25%,50%,75%,max
거리 증가(m),960.0,292.0,354.0,1.0,63.0,109.0,469.0,1473.0


In [10]:
# ============================================================
# 대규모점포: 계약월 1일 기준 영업 중인 점포
# ============================================================
store_raw = pd.read_csv(EXTERNAL_DIR / "large_scale_store.csv", encoding="cp949", low_memory=False, dtype={"관리번호": str})
store_coord = pd.read_csv(EXTERNAL_DIR / "large_store_coords.csv", dtype={"관리번호": str})
seoul = store_raw["도로명주소"].str.contains("서울특별시", na=False) | store_raw["지번주소"].str.contains("서울특별시", na=False)
stores = store_raw[seoul & store_raw["점포구분명"].eq("대규모점포")].merge(
    store_coord[["관리번호", "위도", "경도", "좌표출처"]], on="관리번호", how="left", validate="one_to_one")
for c in ["인허가일자", "폐업일자", "휴업시작일자", "휴업종료일자", "인허가취소일자", "최종수정시점"]:
    stores[c] = pd.to_datetime(stores[c], errors="coerce")
# 상태는 폐업·휴업·취소인데 날짜가 비어 있으면 최종수정시점을 그 날짜로 본다 (그 전에는 영업)
fill = {"폐업": "폐업일자", "휴업": "휴업시작일자", "취소/말소/만료/정지/중지": "인허가취소일자"}
for status, col in fill.items():
    miss = stores["영업상태명"].eq(status) & stores[col].isna()
    print(f"{status} 상태인데 {col} 없음: {miss.sum()}곳 → 최종수정시점으로 보완")
    stores.loc[miss, col] = stores.loc[miss, "최종수정시점"]
print("인허가일자 없음:", stores["인허가일자"].isna().sum())
manual_drop = (stores["사업장명"].eq("통일상가") & stores["도로명주소"].str.contains("장충단로13길 20", na=False)) \
    | stores["사업장명"].eq("(주)공항시장")
stores = stores[~manual_drop]
no_coord = stores[stores["좌표출처"].eq("실패")]
print("좌표 보완 실패 점포:")
display(no_coord[["사업장명", "영업상태명", "인허가일자", "폐업일자"]])
assert (no_coord["폐업일자"] < FIRST_MONTH).all(), "좌표 없는 점포가 분석 기간에 영업 중"
stores = stores[stores["좌표출처"].ne("실패")]


def active_stores(day):
    closed = stores["폐업일자"].notna() & (stores["폐업일자"] <= day)
    cancelled = stores["인허가취소일자"].notna() & (stores["인허가취소일자"] <= day)
    suspended = stores["휴업시작일자"].notna() & (stores["휴업시작일자"] <= day) \
        & (stores["휴업종료일자"].isna() | (stores["휴업종료일자"] > day))
    return stores[(stores["인허가일자"] <= day) & ~closed & ~cancelled & ~suspended]


# 재현 점검: 현재 영업 중(V1 조건) 점포로 계산한 거리 vs V1 대규모점포거리
now = stores[stores["영업상태명"].eq("영업/정상") & stores["상세영업상태명"].eq("정상영업")]
coords["현재점포"] = nearest_m(coords["위도"], coords["경도"], now["위도"], now["경도"])
chk = v1[["위도", "경도", "대규모점포거리_m"]].merge(coords, on=["위도", "경도"], how="left")
chk.loc[chk["현재점포"] < 1, "현재점포"] = 0
gap = (chk["현재점포"] - chk["대규모점포거리_m"]).abs()
print(f"현재 점포 기준 V1 재현: 1m 이내 {(gap < 1).mean():.2%}, 50m 이내 {(gap < 50).mean():.2%}, 최대 {gap.max():.0f}m")
assert (gap < 50).mean() > 0.99

trade_store = v1[["transaction_id", "위도", "경도", "거래월일자"]].copy()
trade_store["대규모점포거리_m"] = np.nan
store_count = {}
for m in MONTHS:
    act = active_stores(m)
    store_count[m] = len(act)
    rows = trade_store["거래월일자"].eq(m)
    trade_store.loc[rows, "대규모점포거리_m"] = nearest_m(trade_store.loc[rows, "위도"], trade_store.loc[rows, "경도"],
                                                       act["위도"], act["경도"])
trade_store.loc[trade_store["대규모점포거리_m"] < 1, "대규모점포거리_m"] = 0
print("월별 영업 점포 수:", min(store_count.values()), "~", max(store_count.values()))
gap_store = (trade_store["대규모점포거리_m"] - v1["대규모점포거리_m"]).abs()
print(f"V1과 50m 넘게 다른 거래: {(gap_store > 50).mean():.2%}")
display(pd.Series(store_count).iloc[::6].to_frame("영업 점포 수").T)

폐업 상태인데 폐업일자 없음: 0곳 → 최종수정시점으로 보완
휴업 상태인데 휴업시작일자 없음: 0곳 → 최종수정시점으로 보완
취소/말소/만료/정지/중지 상태인데 인허가취소일자 없음: 3곳 → 최종수정시점으로 보완
인허가일자 없음: 0
좌표 보완 실패 점포:


,사업장명,영업상태명,인허가일자,폐업일자
13,중학아케이드,폐업,2011-10-07,2014-09-01


현재 점포 기준 V1 재현: 1m 이내 99.83%, 50m 이내 100.00%, 최대 1m


월별 영업 점포 수: 413 ~ 421
V1과 50m 넘게 다른 거래: 5.72%


,2021-07-01,2022-01-01,2022-07-01,2023-01-01,2023-07-01,2024-01-01,2024-07-01,2025-01-01,2025-07-01,2026-01-01
영업 점포 수,416,415,418,418,416,418,417,420,418,415


In [11]:
# ============================================================
# 역 출입구 · 주요 일자리 · 유치원: 거래 시점 기준
# ============================================================
# --- 역 출입구 (개통일은 최근접 역에서 이어받음) ---
entrances = pd.read_csv(EXTERNAL_DIR / "subway_entrances.csv", dtype={"지하철역ID": str, "출입구번호": str})
st_d, st_i = BallTree(np.radians(stations[["역위도", "역경도"]].to_numpy()), metric="haversine").query(
    np.radians(entrances[["위도", "경도"]].to_numpy()), k=1)
entrances["역거리_m"] = st_d[:, 0] * EARTH_RADIUS_M
entrances["역위치"] = st_i[:, 0]
entrances["개통일"] = stations["개통일"].to_numpy()[st_i[:, 0]]
far = entrances["역거리_m"] > 500
print(f"출입구 {len(entrances):,}개 | 최근접 역과 500m 넘게 떨어진 출입구 {far.sum()}개 제외 (좌표 오류 · 역 목록 밖)")
entrances = entrances[~far]
no_exit = stations.iloc[np.setdiff1d(np.arange(len(stations)), entrances["역위치"].unique())]
entry_points = pd.concat([entrances[["위도", "경도", "개통일"]].assign(종류="출입구"),
                          no_exit.rename(columns={"역위도": "위도", "역경도": "경도"})[["위도", "경도", "개통일"]].assign(종류="역 좌표")],
                         ignore_index=True)
print("거리 계산 지점:", entry_points["종류"].value_counts().to_dict(), "(출입구 자료가 없는 역은 역 좌표)")

trade_station["역출입구거리_m"] = np.nan
for start, end in zip([pd.Timestamp.min] + list(cuts), list(cuts) + [pd.Timestamp.max]):
    active = entry_points[entry_points["개통일"].isna() | (entry_points["개통일"] <= start)]
    rows = trade_station["계약일자"].ge(start) & trade_station["계약일자"].lt(end)
    if rows.any():
        trade_station.loc[rows, "역출입구거리_m"] = nearest_m(trade_station.loc[rows, "위도"], trade_station.loc[rows, "경도"],
                                                         active["위도"], active["경도"])
diff = trade_station["역출입구거리_m"] - trade_station["역세권거리_m"]
print("출입구 거리 − 역 좌표 거리 (m):", diff.describe(percentiles=[.1, .5, .9]).round(0).to_dict())
assert trade_station["역출입구거리_m"].notna().all() and diff.median() < 0


def entrance_dist(day, lat, lon):
    act = entry_points[entry_points["개통일"].isna() | (entry_points["개통일"] < day)]
    return nearest_m(lat, lon, act["위도"], act["경도"])


# --- 주요 일자리: 분기별 직장인구 상위 10개 행정동, 분기 시작 + 5개월부터 사용 ---
workplace = pd.read_csv(EXTERNAL_DIR / "workplace_population.csv", encoding="cp949",
                        dtype={"행정동_코드": str, "기준_년분기_코드": str})
admin_area = pd.read_csv(EXTERNAL_DIR / "서울시 상권분석서비스(영역-행정동).csv", encoding="cp949", dtype={"행정동_코드": str})
admin_area["경도"], admin_area["위도"] = Transformer.from_crs("EPSG:5181", "EPSG:4326", always_xy=True).transform(
    admin_area["엑스좌표_값"].to_numpy(), admin_area["와이좌표_값"].to_numpy())
admin_xy = admin_area.set_index("행정동_코드")[["위도", "경도"]]


def top10(df):
    return frozenset(df.groupby("행정동_코드")["총_직장_인구_수"].mean().nlargest(10).index)


def job_dist(top, lat, lon):
    pts = admin_xy.loc[sorted(top)]
    return nearest_m(lat, lon, pts["위도"], pts["경도"])


v1_top = top10(workplace)  # V1 정의: 전체 분기 평균
coords["V1일자리"] = job_dist(v1_top, coords["위도"], coords["경도"])
chk = v1[["위도", "경도", "주요일자리거리_m"]].merge(coords, on=["위도", "경도"], how="left")
assert (chk["V1일자리"] - chk["주요일자리거리_m"]).abs().max() < 1e-6
print("V1 주요일자리거리 재현: 오차 0")

quarters = pd.DataFrame({"분기": sorted(workplace["기준_년분기_코드"].unique())})
quarters["사용가능일"] = [pd.Timestamp(int(q[:4]), 3 * int(q[4]) - 2, 1) + pd.DateOffset(months=5) for q in quarters["분기"]]
quarters["top10"] = [top10(workplace[workplace["기준_년분기_코드"].eq(q)]) for q in quarters["분기"]]
quarters["V1과_같음"] = quarters["top10"].eq(v1_top)
display(quarters.assign(V1과_다른_행정동=[sorted(admin_area.set_index("행정동_코드").loc[sorted(t - v1_top), "행정동_명"])
                                       for t in quarters["top10"]]).drop(columns="top10"))


def job_quarter(day):
    usable = quarters[quarters["사용가능일"] <= day]
    assert len(usable), day
    return usable.iloc[-1]


trade_job = v1[["transaction_id", "위도", "경도", "계약일자"]].copy()
q_dates = quarters["사용가능일"].to_numpy()
trade_job["일자리분기"] = quarters["분기"].to_numpy()[np.searchsorted(q_dates, trade_job["계약일자"].to_numpy(), side="right") - 1]
assert trade_job["계약일자"].min() >= quarters["사용가능일"].min()
trade_job["주요일자리거리_m"] = np.nan
for q, rows in trade_job.groupby("일자리분기").groups.items():
    top = quarters.set_index("분기").loc[q, "top10"]
    trade_job.loc[rows, "주요일자리거리_m"] = job_dist(top, trade_job.loc[rows, "위도"], trade_job.loc[rows, "경도"])
same_q = trade_job["일자리분기"].map(quarters.set_index("분기")["V1과_같음"])
gap = (trade_job["주요일자리거리_m"] - v1["주요일자리거리_m"]).abs()
assert (gap[same_q] < 1e-6).all()  # V1과 같은 목록을 쓰는 거래는 V1 값과 같아야 한다
print(f"주요 일자리 거리가 V1과 달라진 거래: {(gap > 1e-6).sum():,}건 (목록이 바뀐 분기를 쓰는 거래 {(~same_q).sum():,}건 중)")

# --- 유치원: 개원일 이후만 ---
kinder = pd.read_csv(EXTERNAL_DIR / "kindergarten_locations.csv", encoding="cp949", dtype=str)
kcache = pd.read_csv(EXTERNAL_DIR / "kindergarten_geocode_cache.csv").drop_duplicates("검색주소", keep="last")
kinder["검색주소"] = kinder["주소"].str.strip()
kinder = kinder.merge(kcache[["검색주소", "위도", "경도"]], on="검색주소", how="left", validate="many_to_one").dropna(subset=["위도", "경도"])
kinder["개원일"] = pd.to_datetime(kinder["개원일"], format="%Y%m%d")
coords["V1유치원"] = nearest_m(coords["위도"], coords["경도"], kinder["위도"], kinder["경도"])
chk = v1[["위도", "경도", "유치원거리_m"]].merge(coords, on=["위도", "경도"], how="left")
print("V1 유치원거리 재현 최대 오차(m):", float((chk["V1유치원"] - chk["유치원거리_m"]).abs().max()))
assert (chk["V1유치원"] - chk["유치원거리_m"]).abs().max() < 1e-6

k_cuts = sorted(d for d in kinder["개원일"].unique() if d > v1["계약일자"].min())
trade_kinder = v1[["transaction_id", "위도", "경도", "계약일자"]].copy()
trade_kinder["유치원거리_m"] = np.nan
for start, end in zip([pd.Timestamp.min] + k_cuts, k_cuts + [pd.Timestamp.max]):
    rows = trade_kinder["계약일자"].ge(start) & trade_kinder["계약일자"].lt(end)
    if rows.any():
        act = kinder[kinder["개원일"] <= trade_kinder.loc[rows, "계약일자"].min()]  # 구간 안에는 새 개원이 없다
        trade_kinder.loc[rows, "유치원거리_m"] = nearest_m(trade_kinder.loc[rows, "위도"], trade_kinder.loc[rows, "경도"],
                                                       act["위도"], act["경도"])
changed_k = (trade_kinder["유치원거리_m"] - v1["유치원거리_m"]).abs() > 1e-6
print(f"개원 전 유치원이 빠져 거리가 바뀐 거래: {changed_k.sum():,}건 (기간 중 개원 유치원 {len(k_cuts)}개 날짜)")
assert (trade_kinder["유치원거리_m"] >= v1["유치원거리_m"] - 1e-6).all()  # 유치원을 빼기만 하므로 거리는 늘거나 같다


def kinder_dist(day, lat, lon):
    act = kinder[kinder["개원일"] < day]
    return nearest_m(lat, lon, act["위도"], act["경도"])


location_extra = (trade_station[["transaction_id", "역출입구거리_m"]]
                  .merge(trade_job[["transaction_id", "주요일자리거리_m"]], on="transaction_id", validate="one_to_one")
                  .merge(trade_kinder[["transaction_id", "유치원거리_m"]], on="transaction_id", validate="one_to_one"))
assert len(location_extra) == len(v1) and location_extra.notna().all().all()

# 경계일 골든 테스트: 개통 · 개원 · 분기 사용 시작일 당일은 포함, 전날은 제외
assert job_quarter(pd.Timestamp("2025-02-28"))["분기"] == "20243" and job_quarter(pd.Timestamp("2025-03-01"))["분기"] == "20244"
for day in [pd.Timestamp("2025-02-28"), pd.Timestamp("2025-03-01")]:
    rows = trade_job["계약일자"].eq(day)
    expect = job_dist(job_quarter(day)["top10"], trade_job.loc[rows, "위도"], trade_job.loc[rows, "경도"])
    assert rows.any() and np.allclose(trade_job.loc[rows, "주요일자리거리_m"], expect)
checked = 0
for d in k_cuts:
    for day in [d - pd.Timedelta(days=1), d]:
        rows = trade_kinder["계약일자"].eq(day)
        if rows.any():
            act = kinder[kinder["개원일"] <= day]
            assert np.allclose(trade_kinder.loc[rows, "유치원거리_m"],
                               nearest_m(trade_kinder.loc[rows, "위도"], trade_kinder.loc[rows, "경도"], act["위도"], act["경도"]))
            checked += rows.sum()
for d in cuts:
    for day in [d - pd.Timedelta(days=1), d]:
        rows = trade_station["계약일자"].eq(day)
        if rows.any():
            act = entry_points[entry_points["개통일"].isna() | (entry_points["개통일"] <= day)]
            assert np.allclose(trade_station.loc[rows, "역출입구거리_m"],
                               nearest_m(trade_station.loc[rows, "위도"], trade_station.loc[rows, "경도"], act["위도"], act["경도"]))
            checked += rows.sum()
print(f"경계일 골든 테스트 통과: 분기 사용 시작일 2건 · 개원일/개통일 전날·당일 거래 {checked:,}건")

출입구 2,571개 | 최근접 역과 500m 넘게 떨어진 출입구 7개 제외 (좌표 오류 · 역 목록 밖)
거리 계산 지점: {'출입구': 2564, '역 좌표': 492} (출입구 자료가 없는 역은 역 좌표)


출입구 거리 − 역 좌표 거리 (m): {'count': 234302.0, 'mean': -57.0, 'std': 77.0, 'min': -482.0, '10%': -158.0, '50%': -49.0, '90%': 25.0, 'max': 238.0}
V1 주요일자리거리 재현: 오차 0


,분기,사용가능일,V1과_같음,V1과_다른_행정동
0,20211,2021-06-01,True,[]
1,20212,2021-09-01,True,[]
2,20213,2021-12-01,True,[]
3,20214,2022-03-01,True,[]
4,20221,2022-06-01,True,[]
5,20222,2022-09-01,True,[]
6,20223,2022-12-01,True,[]
7,20224,2023-03-01,False,[잠실6동]
8,20231,2023-06-01,False,[잠실6동]
9,20232,2023-09-01,False,[잠실6동]


주요 일자리 거리가 V1과 달라진 거래: 26,466건 (목록이 바뀐 분기를 쓰는 거래 142,574건 중)
V1 유치원거리 재현 최대 오차(m): 0.0


개원 전 유치원이 빠져 거리가 바뀐 거래: 1,275건 (기간 중 개원 유치원 17개 날짜)


경계일 골든 테스트 통과: 분기 사용 시작일 2건 · 개원일/개통일 전날·당일 거래 5,265건


C:\Users\UK\AppData\Local\Temp\ipykernel_311300\1201651335.py:131: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`).Please use a specific unit instead.
  for day in [d - pd.Timedelta(days=1), d]:
C:\Users\UK\AppData\Local\Temp\ipykernel_311300\1201651335.py:131: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`).Please use a specific unit instead.
  for day in [d - pd.Timedelta(days=1), d]:
C:\Users\UK\AppData\Local\Temp\ipykernel_311300\1201651335.py:131: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`).Please use a specific unit instead.
  for day in [d - pd.Timedelta(days=1), d]:
C:\Users\UK\AppData\Local\Te

## 6. 구×월 V2 테이블 (01·03·04·05) + 공급 ÷ 재고

- 각 테이블은 이미 'm월 1일 상태'로 만들어져 있다 (각 노트북 누수 검증 통과).
- 구 단위 공급 원수치는 구 크기에 따라 달라 구 ID처럼 쓰일 수 있으므로 **아파트 재고 1천 호당 비율**을 모델 Feature로 쓴다.
  분모는 05의 `housing_stock_v2` (주택총조사 t년 → t+1년 8월부터). 원수치는 웹 표시·ablation용으로 남긴다.
- 서울 단위 공급은 분모가 거의 일정해 비율을 만들지 않는다.

In [12]:
# ============================================================
# 구×월 테이블 결합
# ============================================================
def load_gm(name):
    d = pd.read_parquet(PROCESSED_DIR / f"{name}.parquet")
    assert len(d) == 1500 and not d.duplicated(["기준월", "자치구"]).any(), name
    return d


market_leading = load_gm("market_leading_v2")
policy = load_gm("policy_v2")
supply = load_gm("supply_v2")
demo = load_gm("demographics_v2")
stock = load_gm("housing_stock_v2")

gm = market_leading
for name, d in [("policy_v2", policy), ("supply_v2", supply), ("demographics_v2", demo)]:
    d = d.drop(columns=["권역"], errors="ignore")
    overlap = (set(gm.columns) & set(d.columns)) - {"기준월", "자치구"}
    assert not overlap, (name, overlap)
    gm = gm.merge(d, on=["기준월", "자치구"], how="inner", validate="one_to_one")
assert len(gm) == 1500

SUPPLY_GU_COUNTS = ["분양공고세대_3m", "분양공고세대_6m", "분양공고세대_12m", "입주예정_공고기준_24m", "미분양_구", "준공후미분양_구"]
gm = gm.merge(stock[["기준월", "자치구", "아파트재고"]], on=["기준월", "자치구"], how="left", validate="one_to_one")
for c in SUPPLY_GU_COUNTS:
    gm[f"{c}_재고천호당"] = gm[c] / gm["아파트재고"] * 1000
SUPPLY_RATIO_COLS = [f"{c}_재고천호당" for c in SUPPLY_GU_COUNTS]
gm = gm.drop(columns="아파트재고")
display(gm[SUPPLY_RATIO_COLS].describe(percentiles=[0.5, 0.9]).round(2).T)

GM_FROM = {}
for name, d in [("01_market", market_leading), ("03_policy", policy), ("04_supply", supply), ("05_demographics", demo)]:
    for c in d.columns:
        if c not in ("기준월", "자치구", "권역"):
            GM_FROM[c] = name
for c in SUPPLY_RATIO_COLS:
    GM_FROM[c] = "06_supply_ratio"
GM_FROM["권역"] = "01_market"
print("구×월 V2 Feature:", len(GM_FROM))

,count,mean,std,min,50%,90%,max
분양공고세대_3m_재고천호당,1500.0,1.01,3.71,0.0,0.00,2.31,53.72
분양공고세대_6m_재고천호당,1500.0,1.99,5.32,0.0,0.00,6.48,53.72
분양공고세대_12m_재고천호당,1500.0,4.15,7.84,0.0,1.29,11.82,59.09
입주예정_공고기준_24m_재고천호당,1500.0,6.67,11.67,0.0,2.29,18.48,85.11
미분양_구_재고천호당,1500.0,0.49,1.06,0.0,0.00,1.53,12.23
준공후미분양_구_재고천호당,1500.0,0.22,0.61,0.0,0.00,0.91,5.13


구×월 V2 Feature: 90


## 7. `v2_trade` (거래 단위, 11 회귀)

| 묶음 | 내용 |
|---|---|
| 키·정답 | `transaction_id`, 단지·구·동, 계약일, `거래금액`·`log_price`·`평당가`·`log_평당가` (정답, Feature 금지) |
| V1 유지 | 전용면적·층·연식·경과월·월 sin/cos·위도·경도·응급의료기관 거리, 거래유형·토지임대부 |
| V1 → 재계산 | 시장(확정월)·전세/월세(확정월)·대출금리(최근공표)·역/점포/주요일자리/유치원 거리(시점) — V1 값은 `_v1` |
| 신규 위치 | 역출입구거리 (출입구 좌표, 개통일 반영) |
| V2 | 02 학교·학원, 03 정책(계약일), 05b 단지 속성·면적 구성·공시·재건축 단계, 구×월(01·03·04·05·공급비율) |
| 제외 | 이름형 범주, V1 정책 5개, km 중복, 주소·코드, 거래 이후 정보(등기일자·해제·매도/매수자·중개사) |

In [13]:
# ============================================================
# 거래 단위 결합
# ============================================================
KEY_COLS = ["transaction_id", "단지일련번호", "시군구명", "법정동", "단지명", "계약일자", "거래월일자"]
TARGET_COLS = ["거래금액", "log_price", "평당가", "log_평당가"]
V1_KEEP = ["전용면적", "층", "연식", "경과월", "월_sin", "월_cos", "위도", "경도",
           "응급의료기관거리_m", "거래유형", "토지임대부아파트여부", "건축년도"]
V1_TO_V1SUFFIX = V1_MARKET_COLS + V1_RENT_COLS + [c for c in V1_RATE_COLS if not c.startswith("기준금리")] \
    + ["역세권거리_m", "대규모점포거리_m", "초등학교거리_m", "중학교거리_m", "고등학교거리_m", "유치원거리_m", "주요일자리거리_m"]
V1_DROPPED_POLICY = ["DSR단계", "스트레스DSR단계", "규제지역여부", "주담대규제강화여부", "고가주택_주담대한도차등여부"]

trade = v1[KEY_COLS + TARGET_COLS + V1_KEEP].copy()
trade["기준월"] = trade["거래월일자"].dt.strftime("%Y-%m")
trade["데이터구분"] = split_of(trade["거래월일자"])
for c in V1_TO_V1SUFFIX:
    trade[f"{c}_v1"] = v1[c].to_numpy()


def attach(left, right, on, name, how="left", validate="many_to_one"):
    overlap = (set(left.columns) & set(right.columns)) - set(on if isinstance(on, list) else [on])
    assert not overlap, (name, overlap)
    out = left.merge(right, on=on, how=how, validate=validate)
    assert len(out) == len(left), name
    return out


trade = attach(trade, market_gu, ["시군구명", "거래월일자"], "market")
trade = attach(trade, rent_trade, "transaction_id", "rent", validate="one_to_one")
trade = attach(trade, rates[["거래월일자"] + RATE_COLS], "거래월일자", "rates")
trade = attach(trade, trade_station[["transaction_id", "역세권거리_m"]], "transaction_id", "station", validate="one_to_one")
trade = attach(trade, trade_store[["transaction_id", "대규모점포거리_m"]], "transaction_id", "store", validate="one_to_one")
trade = attach(trade, location_extra, "transaction_id", "location_extra", validate="one_to_one")

school = pd.read_parquet(PROCESSED_DIR / "school_environment_v2.parquet")
policy_trade = pd.read_parquet(PROCESSED_DIR / "policy_trade_v2.parquet")
complex_trade = pd.read_parquet(PROCESSED_DIR / "complex_trade_v2.parquet")
attrs = pd.read_parquet(PROCESSED_DIR / "complex_attributes_v2.parquet").drop(columns=["lot"])
unit_mix = pd.read_parquet(PROCESSED_DIR / "complex_unit_mix_v2.parquet")
for d in (attrs, unit_mix):
    d["단지일련번호"] = d["단지일련번호"].astype(str)

trade = attach(trade, school, "transaction_id", "school", validate="one_to_one")
trade = attach(trade, policy_trade, "transaction_id", "policy_trade", validate="one_to_one")
# 예상가격은 단지 이력이 없으면 동·면적대 → 구 중위값으로 대체된다. 신축은 주변 구축보다 비싸서
# 대체값이 기준가를 잘못 준다 (11 1단계: Valid 신규 단지 MAPE 14.3 → 9.4%) → 모델에는 단지 자체 이력 값만 쓴다.
trade["예상가격_단지만"] = trade["예상가격_만원"].where(trade["예상가격_출처"].eq("단지"))
trade = attach(trade, complex_trade, "transaction_id", "complex_trade", validate="one_to_one")
trade = attach(trade, attrs, "단지일련번호", "complex_attrs")
trade = attach(trade, unit_mix, "단지일련번호", "unit_mix")
trade = attach(trade, gm.drop(columns=["권역"]).rename(columns={"자치구": "시군구명"}), ["기준월", "시군구명"], "region_month")
trade = trade.sort_values("transaction_id").reset_index(drop=True)
print("v2_trade:", trade.shape)
assert trade["transaction_id"].is_unique and len(trade) == len(v1)

v2_trade: (234302, 279)


## 8. `v2_region_month` (구 × 예측시점, 31 분류)

- 행 = (구, **예측시점 p**). p월 1일에 예측한다고 본다.
- 정답 = V1과 같은 정의: 향후 3개월(p, p+1, p+2) 월 중위평당가의 중위 ÷ 직전 3개월(p-3~p-1) 월 중위평당가의 중위 − 1 > 3%.
  V1의 기준월 m은 p = m + 1과 같다 (V1은 m월 말에 m월 거래까지 다 안다고 가정했다).
- V1 분류 피처(가격 lag·변화율·변동성·거래량·서울 국면·직전확정 변화율)는 같은 계산을 **p-2월(확정월)까지**의 거래로 다시 만든다.
  V1은 기준월 m의 당월 거래를 썼는데, m월 거래는 m+1월 말에야 다 신고되므로 실제 예측 시점에는 알 수 없는 정보였다.
- 정답 계산에는 워밍업 거래도 써서 2021-07~09 예측시점의 정답도 만든다 (V1 기간과 겹치는 행은 V1 정답과 같아야 한다).

In [14]:
# ============================================================
# 구×예측시점 정답
# ============================================================
gu_month, _ = monthly_market(pool)
gu_month = gu_month.sort_values(["시군구명", "거래월일자"]).reset_index(drop=True)
g = gu_month.groupby("시군구명")["월중위평당가"]
past3 = g.transform(lambda s: s.shift(1).rolling(3, min_periods=3).median())          # p-3 ~ p-1
future = pd.concat([g.shift(-k) for k in range(3)], axis=1)                              # p, p+1, p+2
target = pd.DataFrame({"자치구": gu_month["시군구명"], "예측시점": gu_month["거래월일자"]})
ok = past3.notna() & future.notna().all(axis=1)
target["향후3개월_변화율"] = np.where(ok, future.median(axis=1) / past3 - 1, np.nan)
target["상승여부"] = np.where(ok, (target["향후3개월_변화율"] > 0.03).astype(float), np.nan)

# V1 정답 재현 (V1 30_classification 정의, 기준월 m) → p = m + 1 과 비교
v1_gm = v1.groupby(["시군구명", "거래월일자"]).agg(월중위평당가=("평당가", "median")).reset_index().sort_values(["시군구명", "거래월일자"])
gg = v1_gm.groupby("시군구명")["월중위평당가"]
v1_gm["최근3개월"] = gg.transform(lambda x: x.rolling(3, min_periods=3).median())
fut = pd.concat([gg.shift(-k) for k in [1, 2, 3]], axis=1)
okv = v1_gm["최근3개월"].notna() & fut.notna().all(axis=1)
v1_gm["V1_변화율"] = np.where(okv, fut.median(axis=1) / v1_gm["최근3개월"] - 1, np.nan)
v1_gm["예측시점"] = v1_gm["거래월일자"] + pd.DateOffset(months=1)
cmp = v1_gm.dropna(subset=["V1_변화율"]).merge(target, left_on=["시군구명", "예측시점"], right_on=["자치구", "예측시점"])
print("V1 정답과 겹치는 행:", len(cmp), "| 최대 차이:", float((cmp["V1_변화율"] - cmp["향후3개월_변화율"]).abs().max()))
assert len(cmp) == okv.sum() and (cmp["V1_변화율"] - cmp["향후3개월_변화율"]).abs().max() < 1e-12
print("V1 상승 비율:", round(float((cmp["V1_변화율"] > 0.03).mean()), 4))

target = target[target["예측시점"].between(FIRST_MONTH, LAST_MONTH)].reset_index(drop=True)
print("정답이 있는 예측시점:", target.dropna()["예측시점"].min().date(), "~", target.dropna()["예측시점"].max().date(),
      f"({target['상승여부'].notna().sum()}행, 상승 {target['상승여부'].mean():.1%})")

V1 정답과 겹치는 행: 1375 | 최대 차이: 0.0
V1 상승 비율: 0.3731
정답이 있는 예측시점: 2021-07-01 ~ 2026-04-01 (1450행, 상승 39.4%)


In [15]:
# ============================================================
# 구×예측시점 시장 피처: V1 분류 피처 정의를 확정월(p-2)까지의 거래로
# ============================================================
def classification_market(trades, lag=TRADE_LAG):
    gm_, seoul_ = monthly_market(trades)
    gm_ = gm_.sort_values(["시군구명", "거래월일자"]).reset_index(drop=True)
    f = gm_[["시군구명", "거래월일자"]].copy()
    g = gm_.groupby("시군구명")
    p, n = gm_["월중위평당가"], gm_["월거래량"]
    # V1 정의를 '각 월 시점'에서 계산한 뒤 lag만큼 뒤로 민다: 행 p는 p-lag월까지의 거래만 반영
    base = pd.DataFrame({"시군구명": gm_["시군구명"]})
    base["평당가"] = p
    for k in [1, 3, 6]:
        base[f"평당가_{k}개월변화율"] = p / g["월중위평당가"].shift(k) - 1
    base["중위평당가_3개월"] = g["월중위평당가"].transform(lambda s: s.rolling(3, min_periods=3).median())
    base["중위평당가_6개월"] = g["월중위평당가"].transform(lambda s: s.rolling(6, min_periods=6).median())
    base["가격변동성_6개월"] = g["월중위평당가"].transform(lambda s: s.rolling(6, min_periods=6).std())
    base["거래량"] = n
    base["평균거래량_3개월"] = g["월거래량"].transform(lambda s: s.rolling(3, min_periods=3).mean())
    base["평균거래량_6개월"] = g["월거래량"].transform(lambda s: s.rolling(6, min_periods=6).mean())
    base["거래량_전월대비"] = n / g["월거래량"].shift(1) - 1
    base["거래량_3개월평균대비"] = n / base["평균거래량_3개월"] - 1
    # 실현된 과거 변화율: 창 끝이 확정월인 '향후 3개월 변화율' (V1 직전확정_3개월변화율과 같은 개념)
    past3 = g["월중위평당가"].transform(lambda s: s.shift(3).rolling(3, min_periods=3).median())
    last3 = g["월중위평당가"].transform(lambda s: s.rolling(3, min_periods=3).median())
    base["실현_3개월변화율"] = last3 / past3 - 1
    cols = [c for c in base.columns if c != "시군구명"]
    shifted = base.groupby("시군구명")[cols].shift(lag)
    f[[f"구_확정월기준_{c}" for c in cols]] = shifted.to_numpy()

    s = seoul_.copy()
    sm = pd.DataFrame({"거래월일자": s["거래월일자"]})
    sp = gm_.groupby("거래월일자")["월중위평당가"].median().reindex(s["거래월일자"]).to_numpy()  # V1: 구 월중위평당가의 중위
    sp = pd.Series(sp)
    for k in [1, 3, 6]:
        sm[f"서울_{k}개월변화율"] = sp / sp.shift(k) - 1
    sm["서울_거래량"] = gm_.groupby("거래월일자")["월거래량"].sum().reindex(s["거래월일자"]).to_numpy()
    ch3 = (p / g["월중위평당가"].shift(3) - 1)
    breadth = pd.DataFrame({"거래월일자": gm_["거래월일자"], "ch3": ch3}).groupby("거래월일자")["ch3"]
    sm["상승구비율"] = breadth.apply(lambda x: (x > 0).mean() if x.notna().all() else np.nan).reindex(s["거래월일자"]).to_numpy()
    sm["강한상승구비율"] = breadth.apply(lambda x: (x > 0.03).mean() if x.notna().all() else np.nan).reindex(s["거래월일자"]).to_numpy()
    sm["강한하락구비율"] = breadth.apply(lambda x: (x < -0.03).mean() if x.notna().all() else np.nan).reindex(s["거래월일자"]).to_numpy()
    scols = [c for c in sm.columns if c != "거래월일자"]
    sm[[f"서울_확정월기준_{c.replace('서울_', '')}" for c in scols]] = sm[scols].shift(lag).to_numpy()
    sm = sm.drop(columns=scols)
    return f.merge(sm, on="거래월일자", how="left").rename(columns={"시군구명": "자치구", "거래월일자": "예측시점"})


cls_market = classification_market(pool)
CLS_MARKET_COLS = [c for c in cls_market.columns if c not in ("자치구", "예측시점")]
cls_market = cls_market[cls_market["예측시점"].between(FIRST_MONTH, LAST_MONTH)].reset_index(drop=True)
display(cls_market[CLS_MARKET_COLS].isna().sum().to_frame("결측").T)

# 잘라내기: p-1·p월 거래를 구마다 가짜 1건(1원)으로 바꿔 다시 계산 → 값이 같아야 한다
def cls_truncation_bad(lag):
    full = classification_market(pool, lag).set_index(["자치구", "예측시점"])
    bad = 0
    for m in MONTHS[::3]:
        known = pool[pool["거래월일자"] <= m - pd.DateOffset(months=TRADE_LAG)]
        fake = placeholder([m - pd.DateOffset(months=1), m])
        part = classification_market(pd.concat([known, fake]), lag).set_index(["자치구", "예측시점"])
        a = full.xs(m, level="예측시점")[CLS_MARKET_COLS]
        b = part.xs(m, level="예측시점")[CLS_MARKET_COLS].reindex(a.index)
        bad += int((~np.isclose(a.to_numpy(float), b.to_numpy(float), equal_nan=True)).any(axis=1).sum())
    return bad


bad = cls_truncation_bad(TRADE_LAG)
print("p-1월 이후 거래를 가짜 값으로 바꿨을 때 달라진 구×예측시점:", bad)
assert bad == 0
bad_leaky = cls_truncation_bad(1)
print("시험의 시험 — 시차 1이면 잡히는 행:", bad_leaky)
assert bad_leaky > 0

,구_확정월기준_평당가,구_확정월기준_평당가_1개월변화율,구_확정월기준_평당가_3개월변화율,구_확정월기준_평당가_6개월변화율,구_확정월기준_중위평당가_3개월,구_확정월기준_중위평당가_6개월,구_확정월기준_가격변동성_6개월,구_확정월기준_거래량,구_확정월기준_평균거래량_3개월,구_확정월기준_평균거래량_6개월,구_확정월기준_거래량_전월대비,구_확정월기준_거래량_3개월평균대비,구_확정월기준_실현_3개월변화율,서울_확정월기준_1개월변화율,서울_확정월기준_3개월변화율,서울_확정월기준_6개월변화율,서울_확정월기준_거래량,서울_확정월기준_상승구비율,서울_확정월기준_강한상승구비율,서울_확정월기준_강한하락구비율
결측,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


p-1월 이후 거래를 가짜 값으로 바꿨을 때 달라진 구×예측시점: 0


시험의 시험 — 시차 1이면 잡히는 행: 500


In [16]:
# ============================================================
# v2_region_month 결합
# ============================================================
rent_gu_cols = [c for c in rent_gu.columns if c.startswith("자치구_확정") and not c.endswith("매매평당가")]
rent_gm = rent_gu[["sggCd", "거래월일자"] + rent_gu_cols].copy()
rent_gm["자치구"] = rent_gm["sggCd"].map(GU_NAME)
rent_gm = rent_gm.rename(columns={"거래월일자": "예측시점", **{c: c.replace("자치구_", "구_") for c in rent_gu_cols}})
RENT_GM_COLS = [c.replace("자치구_", "구_") for c in rent_gu_cols]

region = target.merge(cls_market, on=["자치구", "예측시점"], how="left", validate="one_to_one")
region = region.merge(rent_gm[["자치구", "예측시점"] + RENT_GM_COLS], on=["자치구", "예측시점"], how="left", validate="one_to_one")
region = region.merge(rates.rename(columns={"거래월일자": "예측시점"}), on="예측시점", how="left", validate="many_to_one")
region["기준월"] = region["예측시점"].dt.strftime("%Y-%m")
region = region.merge(gm, on=["기준월", "자치구"], how="left", validate="one_to_one")
region["데이터구분"] = split_of(region["예측시점"])
region = region.sort_values(["예측시점", "자치구"]).reset_index(drop=True)
assert len(region) == 1500 and not region.duplicated(["자치구", "예측시점"]).any()
print("v2_region_month:", region.shape)
display(region.groupby("데이터구분")["상승여부"].agg(["count", "mean"]).round(3))

v2_region_month: (1500, 140)


,count,mean
데이터구분,,
test,100,0.500
train,1200,0.388
valid,150,0.373


## 9. `v2_complex` (단지, 21 군집 + 웹)

- 고정 속성: 05b 건축물대장(동수·세대수·층·승강기·주차·용적률·건폐율), 면적 구성(공시가격 전체 호 기준), 위치.
- 시간에 따라 바뀌는 값은 두 시점으로 둔다.
  - `_학습기준`: 2025-07-01에 알 수 있던 값 (학습 구간 끝). 군집 라벨을 회귀에 넣어 보는 실험에서 미래정보가 섞이지 않게 한다.
  - `_최신`: 지금 알 수 있는 값 (웹 표시).
- 위치(시점): 역 · 역 출입구 · 대규모점포 · 주요 일자리 · 유치원 거리를 기준일에 운영 중이던 시설(일자리는 사용 가능한 분기)로 계산.
- 학교: 02 위치 테이블의 2025 학교기준연도 (6월 이후는 당해연도 → 2025-07-01 기준도 2025).
- 학원: 02c와 같은 업종·반경 규칙. 기준일에 쓸 수 있던 정상 상가정보 스냅샷(기준월 + 3개월 ≤ 기준일)
  → 학습기준 202412, 최신 202603. 학원가 거리는 02c의 2020-12 학원가 위치 기준(고정).
- 공시가격: 학습기준 = 2024년 공시(2025년 공시는 2025-07-01부터), 최신 = 2026년 공시.
- 재건축 단계: 05b 구역 고시일로 기준일 이전 가장 높은 단계.

In [17]:
# ============================================================
# 단지 테이블
# ============================================================
SNAP_TRAIN, SNAP_LATEST = TRAIN_END, pd.Timestamp("2026-07-01")

cx = (v1.groupby("단지일련번호")
      .agg(단지명=("단지명", "first"), 시군구명=("시군구명", "first"), 법정동=("법정동", "first"),
           건축년도=("건축년도", "min"),
           위도=("위도", lambda s: s.mode().iloc[0]), 경도=("경도", lambda s: s.mode().iloc[0]),
           응급의료기관거리_m=("응급의료기관거리_m", "median"),
           거래건수_학습기간=("거래월일자", lambda s: int((s < TRAIN_END).sum())),
           첫거래월=("거래월일자", "min"), 마지막거래월=("거래월일자", "max"))
      .reset_index())
cx["단지일련번호"] = cx["단지일련번호"].astype(str)
cx = cx.merge(attrs, on="단지일련번호", how="left", validate="one_to_one")
cx = cx.merge(unit_mix, on="단지일련번호", how="left", validate="one_to_one")

for tag, day in [("학습기준", SNAP_TRAIN), ("최신", SNAP_LATEST)]:
    act_st = stations[stations["개통일"].isna() | (stations["개통일"] < day)]
    cx[f"역세권거리_m_{tag}"] = nearest_m(cx["위도"], cx["경도"], act_st["역위도"], act_st["역경도"])
    cx[f"역출입구거리_m_{tag}"] = entrance_dist(day, cx["위도"], cx["경도"])
    cx[f"주요일자리거리_m_{tag}"] = job_dist(job_quarter(day)["top10"], cx["위도"], cx["경도"])
    cx[f"유치원거리_m_{tag}"] = kinder_dist(day, cx["위도"], cx["경도"])
    act = active_stores(day) if tag == "학습기준" else now
    d = nearest_m(cx["위도"], cx["경도"], act["위도"], act["경도"])
    cx[f"대규모점포거리_m_{tag}"] = np.where(d < 1, 0, d)

# 학교 (02 위치 테이블, 2025)
loc_school = pd.read_parquet(PROCESSED_DIR / "school" / "school_features_by_location_2021_2025.parquet")
sch = loc_school[loc_school["학교기준연도"].eq(2025)].drop(columns=["학교기준연도"])
SCHOOL_NAME_COLS = ["최근접초_학교명", "최근접중_학교명"]
cx = cx.merge(sch, on=["위도", "경도"], how="left", validate="many_to_one")
print("학교 테이블이 붙은 단지:", f"{cx['최근접초_거리_m'].notna().mean():.1%}")

# 학원 (02c와 같은 규칙: 업종 그룹·반경, 스냅샷은 기준월 + 3개월부터 사용, 결손·좌표 오류 분기 제외)
ACADEMY_GROUPS = {"입시교과": ["P10501"], "예체능": ["P10601", "P10609", "P10611", "P10613"], "외국어": ["P10615"]}
ACADEMY_RADIUS = [("입시교과", 500, "반경500m_입시교과학원수"), ("입시교과", 1000, "반경1km_입시교과학원수"),
                  ("예체능", 1000, "반경1km_예체능학원수"), ("외국어", 1000, "반경1km_외국어학원수")]
ACADEMY_STATIC = ["최근접학원가_거리_m", "최근접3대학원가_거리_m", "최근접학원가명", "최근접3대학원가명"]
store_edu = pd.read_parquet(PROCESSED_DIR / "academy" / "store_education_2020_2026.parquet")
academy_trade = (pd.read_parquet(PROCESSED_DIR / "df_fe_academy_v2.parquet")
                 .merge(v1[["transaction_id", "위도", "경도", "거래월일자"]], on="transaction_id", validate="one_to_one"))
normal_snaps = sorted(academy_trade["학원스냅샷기준월"].unique())  # 02c가 정상으로 판정해 거래에 쓴 스냅샷


def academy_counts(snapshot_month, lat, lon):
    snap = store_edu[store_edu["기준월"].eq(snapshot_month)]
    query = np.radians(np.column_stack([lat, lon]))
    out = {}
    for group, radius_m, name in ACADEMY_RADIUS:
        acad = snap[snap["상권업종소분류코드"].isin(ACADEMY_GROUPS[group])]
        tree = BallTree(np.radians(acad[["위도", "경도"]].to_numpy()), metric="haversine")
        out[name] = tree.query_radius(query, r=radius_m / 6_371_000, count_only=True)
    return pd.DataFrame(out)


static = academy_trade.drop_duplicates(["위도", "경도"])[["위도", "경도"] + ACADEMY_STATIC]
cx = cx.merge(static, on=["위도", "경도"], how="left", validate="many_to_one")
for tag, day in [("학습기준", SNAP_TRAIN), ("최신", SNAP_LATEST)]:
    snap = max(m for m in normal_snaps if pd.Period(m, "M") + 3 <= day.to_period("M"))
    counts = academy_counts(snap, cx["위도"].to_numpy(), cx["경도"].to_numpy())
    for _, _, name in ACADEMY_RADIUS:
        cx[f"{name}_{tag}"] = counts[name].to_numpy()
    # 재현 점검: 같은 스냅샷을 쓴 거래(기준일 달 = 학습기준 2025-07, 최신은 마지막 거래월)의 02c 값과 같아야 한다
    ref = academy_trade[academy_trade["학원스냅샷기준월"].eq(snap)].drop_duplicates(["위도", "경도"])
    chk = ref.merge(cx[["위도", "경도"] + [f"{n}_{tag}" for _, _, n in ACADEMY_RADIUS]], on=["위도", "경도"])
    for _, _, name in ACADEMY_RADIUS:
        assert chk[name].eq(chk[f"{name}_{tag}"]).all(), (tag, name)
    print(f"학원 {tag}: 스냅샷 {snap} | 02c 거래 값과 일치하는 좌표 {len(chk):,}곳")
assert cx[ACADEMY_STATIC].notna().all().all()

# 공시가격
official = pd.read_parquet(PROCESSED_DIR / "complex_official_price_v2.parquet")
official["단지일련번호"] = official["단지일련번호"].astype(str)
for tag, year in [("학습기준", 2024), ("최신", 2026)]:
    cur = official[official["기준연도"].eq(year)].set_index("단지일련번호")["공시_m2당_단지"]
    prev = official[official["기준연도"].eq(year - 1)].set_index("단지일련번호")["공시_m2당_단지"]
    cx[f"공시_m2당_{tag}"] = cx["단지일련번호"].map(cur).astype(float)
    cx[f"공시_전년대비_{tag}"] = ((cx["단지일련번호"].map(cur) / cx["단지일련번호"].map(prev) - 1) * 100).astype(float)

# 재건축 단계
zone = pd.read_parquet(PROCESSED_DIR / "complex_rebuild_zone_v2.parquet")
zone["단지일련번호"] = zone["단지일련번호"].astype(str)
STAGES = [("구역지정_최초", 1), ("추진위원회", 2), ("조합설립", 3), ("사업시행_최초", 4),
          ("관리처분_최초", 5), ("이주시작", 6), ("착공", 7)]
z = cx[["단지일련번호"]].merge(zone, on="단지일련번호", how="left")
for tag, day in [("학습기준", SNAP_TRAIN), ("최신", SNAP_LATEST)]:
    stage = np.zeros(len(z), dtype=int)
    for col, value in STAGES:
        stage = np.where(z[col].notna() & (z[col] < day), np.maximum(stage, value), stage)
    cx[f"재건축_단계_{tag}"] = stage

# 재현 점검: 학습기준 재건축 단계는 2025-06-30 거래의 complex_trade 값과 같아야 한다
last_train = (trade[trade["계약일자"] < TRAIN_END].sort_values("계약일자").groupby("단지일련번호").tail(1)
              [["단지일련번호", "계약일자", "재건축_단계"]])
chk = last_train.merge(cx[["단지일련번호", "재건축_단계_학습기준"]], on="단지일련번호")
same_window = chk["계약일자"] >= "2025-06-01"
mism = (chk.loc[same_window, "재건축_단계"] != chk.loc[same_window, "재건축_단계_학습기준"]).sum()
print("2025-06 마지막 거래 단계와 학습기준 단계가 다른 단지:", mism, "/", same_window.sum())
assert mism <= 0.01 * same_window.sum()

cx = cx.sort_values("단지일련번호").reset_index(drop=True)
assert cx["단지일련번호"].is_unique and len(cx) == v1["단지일련번호"].nunique()
print("v2_complex:", cx.shape)

학교 테이블이 붙은 단지: 100.0%


학원 학습기준: 스냅샷 202412 | 02c 거래 값과 일치하는 좌표 4,760곳
학원 최신: 스냅샷 202603 | 02c 거래 값과 일치하는 좌표 2,337곳


2025-06 마지막 거래 단계와 학습기준 단계가 다른 단지: 1 / 2810
v2_complex: (7871, 80)


## 10. Feature 레지스트리와 최종 검증

역할: `key` 식별·결합 / `target` 정답 (Feature 금지) / `model` 모델 Feature /
`ablation` 11·31에서 넣고 빼며 비교 (출처 플래그, 구 공급 원수치) / `v1_compare` V1 원래 값 (비교용) /
`display` 웹 표시 (이름 등) / `meta` 시점 메타데이터 (모델 금지).

In [18]:
# ============================================================
# 레지스트리
# ============================================================
FORBIDDEN = {"거래금액", "log_price", "평당가", "log_평당가", "향후3개월_변화율", "상승여부",
             "등기일자", "해제여부", "해제사유발생일", "매도자", "매수자", "중개사소재지"}
# 응급의료기관 · 유치원 거리: 원천이 현재 목록이라 과거 시점을 보장할 수 없다 (유치원은 폐원 기관 누락) → 기본 Feature에서 빼고 11에서 비교
# 예상가격 · 주담대 한도: 대체값(동·면적대 · 구)이 섞여 신축 단지를 크게 틀린다 → 예상가격_단지만으로 대체 (11 1단계)
EXPECT_FALLBACK = {"예상가격_만원", "주담대_최대한도_만원", "주담대_한도비율"}
CURRENT_LIST = {"응급의료기관거리_m", "유치원거리_m", "유치원거리_m_학습기준"}
ABLATION = set(SUPPLY_GU_COUNTS) | {c for c in RENT_COLS if c.startswith("전세피처출처")} | {"예상가격_출처"} | CURRENT_LIST | EXPECT_FALLBACK
DISPLAY = {"단지명", "토허_근거", "최근접초등_학교명", "최근접중학_학교명", "최근접학원가명", "최근접3대학원가명"} | set(SCHOOL_NAME_COLS)
META = {"계약일자", "거래월일자", "기준월", "예측시점", "데이터구분", "학교기준연도", "공시연도", "학원스냅샷기준월",
        "학원스냅샷_경과개월", "사용승인연도", "면적기준연도", "건축년도", "첫거래월", "마지막거래월", "거래건수_학습기간", "법정동"}
KEYS = {"transaction_id", "단지일련번호", "시군구명", "자치구"}


LOCATION_PIT = ("역세권거리_m", "대규모점포거리_m", "역출입구거리_m", "주요일자리거리_m", "유치원거리_m")


def source_of(c, dataset):
    if c in KEYS or c in META:
        return "key/meta"
    if c == "응급의료기관거리_m":
        return "V1(현재 목록)"
    if c.endswith("_v1") or c in ("위도", "경도"):
        return "V1"
    if c in GM_FROM:
        return GM_FROM[c]
    if c in MARKET_COLS or c in CLS_MARKET_COLS:
        return "06_market(확정월)"
    if c in RENT_COLS or c in RENT_GM_COLS:
        return "06_rent(확정월)"
    if c in RATE_COLS:
        return "06_rate"
    if c in LOCATION_PIT or c.startswith(tuple(f"{p}_" for p in LOCATION_PIT)):
        return "06_location(시점)"
    if c.startswith(tuple(n for _, _, n in ACADEMY_RADIUS)) and dataset == "v2_complex":
        return "02c_academy(시점)"
    if c in school.columns or c in sch.columns:
        return "02_school"
    if c in policy_trade.columns or c == "예상가격_단지만":
        return "03_policy_trade"
    if c in complex_trade.columns or c in attrs.columns or c in unit_mix.columns or c.startswith(("공시_", "재건축_")):
        return "05b_complex"
    if c in TARGET_COLS or c in ("향후3개월_변화율", "상승여부"):
        return "target"
    return "V1"


def role_of(c):
    if c in FORBIDDEN:
        return "target"
    if c in KEYS:
        return "key"
    if c in META:
        return "meta"
    if c.endswith("_v1"):
        return "v1_compare"
    if c in DISPLAY or c.endswith(("학교명", "학원가명", "_최신")):
        return "display"
    if c in ABLATION:
        return "ablation"
    return "model"


TIMING = {"V1": "고정(위치·물리) 또는 계약일", "V1(현재 목록)": "현재 지정 목록 — 과거 시점 미반영 (ablation)", "06_market(확정월)": "m-2월까지 실거래", "06_rent(확정월)": "m-2월까지 전월세",
          "06_rate": "기준금리 m-1, 대출금리 m-2", "06_location(시점)": "역·출입구 개통일 ≤ 계약일, 점포 계약월 1일 영업, 주요 일자리 분기 시작+5개월 ≤ 계약일, 유치원 개원일 ≤ 계약일",
          "01_market": "부동산원 m-2, 한은 m-1", "03_policy": "발표 < m월 1일", "03_policy_trade": "발표 < 계약일",
          "04_supply": "공표 M+1 → m=M+2", "06_supply_ratio": "공급 M+2, 재고 t년 → t+1년 8월",
          "05_demographics": "공표 M+1 → m=M+2", "02_school": "6월 이후 당해연도 학교 통계, 학원 스냅샷 기준월+3개월",
          "02c_academy(시점)": "상가정보 기준월 + 3개월 ≤ 기준일 (정상 스냅샷)",
          "05b_complex": "공시 Y년 → Y년 7월, 정비 고시일 < 계약일, 대장은 고정 속성", "target": "-", "key/meta": "-"}

rows = []
for dataset, frame in [("v2_trade", trade), ("v2_region_month", region), ("v2_complex", cx)]:
    for c in frame.columns:
        src = source_of(c, dataset)
        rows.append({"dataset": dataset, "column": c, "role": role_of(c), "source": src, "timing": TIMING.get(src, ""),
                     "dtype": str(frame[c].dtype), "na_rate": round(float(frame[c].isna().mean()), 4)})
registry = pd.DataFrame(rows)
display(registry.pivot_table(index="source", columns=["dataset", "role"], values="column", aggfunc="count", fill_value=0))

dataset         v2_complex                        v2_region_month           \
role              ablation display key meta model        ablation key meta   
source                                                                       
01_market                0       0   0    0     0               0   0    0   
02_school                0       4   0    0    25               0   0    0   
02c_academy(시점)          0       4   0    0     4               0   0    0   
03_policy                0       0   0    0     0               0   0    0   
03_policy_trade          0       0   0    0     0               0   0    0   
04_supply                0       0   0    0     0               6   0    0   
05_demographics          0       0   0    0     0               0   0    0   
05b_complex              0       3   0    0    17               0   0    0   
06_location(시점)          1       5   0    0     4               0   0    0   
06_market(확정월)           0       0   0    0     0               0   0    0   
06_rate                  0       0   0    0     0               0   0    0   
06_rent(확정월)             0       0   0    0     0               0   0    0   
06_supply_ratio          0       0   0    0     0               0   0    0   
V1                       0       1   0    0     2               0   0    0   
V1(현재 목록)                1       0   0    0     0               0   0    0   
key/meta                 0       0   2    7     0               0   1    3   
target                   0       0   0    0     0               0   0    0   

dataset                      v2_trade                                           
role            model target ablation display key meta model target v1_compare  
source                                                                          
01_market          17      0        0       0   0    0    16      0          0  
02_school           0      0        0       4   0    0    29      0          0  
02c_academy(시점)     0      0        0       0   0    0     0      0          0  
03_policy          32      0        0       0   0    0    32      0          0  
03_policy_trade     0      0        4       1   0    0     9      0          0  
04_supply          14      0        6       0   0    0    14      0          0  
05_demographics    15      0        0       0   0    0    15      0          0  
05b_complex         0      0        0       0   0    0    21      0          0  
06_location(시점)     0      0        1       0   0    0     4      0          0  
06_market(확정월)     20      0        0       0   0    0    14      0          0  
06_rate             9      0        0       0   0    0     9      0          0  
06_rent(확정월)       15      0        3       0   0    0    15      0          0  
06_supply_ratio     6      0        0       0   0    0     6      0          0  
V1                  0      0        0       1   0    0    10      0         45  
V1(현재 목록)           0      0        1       0   0    0     0      0          0  
key/meta            0      0        0       0   3   12     0      0          0  
target              0      2        0       0   0    0     0      4          0

In [19]:
# ============================================================
# 최종 검증
# ============================================================
problems = []
for dataset, frame in [("v2_trade", trade), ("v2_region_month", region), ("v2_complex", cx)]:
    reg = registry[registry["dataset"].eq(dataset)]
    model_cols = reg.loc[reg["role"].eq("model"), "column"].tolist()
    # 1) 금지 열·이름형 열이 모델 Feature에 없음
    bad = [c for c in model_cols if c in FORBIDDEN or c.startswith("최근접") and frame[c].dtype == object]
    if bad:
        problems.append((dataset, "금지/이름형", bad))
    # 2) 열 이름 중복 없음
    if frame.columns.duplicated().any():
        problems.append((dataset, "중복 열", frame.columns[frame.columns.duplicated()].tolist()))
    # 3) 모델 Feature 중 문자열형은 의도한 범주만
    obj = [c for c in model_cols if frame[c].dtype == object or str(frame[c].dtype) in ("string", "category")]
    print(f"[{dataset}] 행 {len(frame):,} | 열 {frame.shape[1]} | 모델 Feature {len(model_cols)} | 범주형 {obj}")
assert not problems, problems
# V1 정책 5개 열은 V1 값이 아니라 03 정책(같은 이름의 V2 정의)에서만 와야 한다
v1_policy_left = registry[registry["column"].isin(V1_DROPPED_POLICY)]
print("V1 정책 열 이름이 남은 곳 (출처 확인):", v1_policy_left[["dataset", "column", "source"]].values.tolist())
assert v1_policy_left["source"].eq("03_policy").all()

# 4) 결측: 구간별 (모델 Feature, 결측률 상위)
trade_model = registry.loc[registry["dataset"].eq("v2_trade") & registry["role"].eq("model"), "column"].tolist()
na_split = trade.groupby("데이터구분")[trade_model].apply(lambda d: d.isna().mean()).T
na_split = na_split[na_split.max(axis=1) > 0].sort_values("train", ascending=False)
print("\nv2_trade 결측이 있는 모델 Feature:", len(na_split))
display(na_split.head(25).round(4))

# 5) 학습 구간 상수 Feature (트리가 배울 수 없음)
train = trade[trade["데이터구분"].eq("train")]
const_trade = [c for c in trade_model if train[c].nunique(dropna=True) <= 1]
region_model = registry.loc[registry["dataset"].eq("v2_region_month") & registry["role"].eq("model"), "column"].tolist()
region_train = region[region["데이터구분"].eq("train")]
const_region = [c for c in region_model if region_train[c].nunique(dropna=True) <= 1]
print("학습 구간 상수 — v2_trade:", const_trade, "| v2_region_month:", const_region)
registry["train_constant"] = (registry["dataset"].eq("v2_trade") & registry["column"].isin(const_trade)) \
    | (registry["dataset"].eq("v2_region_month") & registry["column"].isin(const_region))

# 6) 확정월 재계산이 V1 값과 얼마나 다른가 (같은 정의, 시차만 다름)
pairs = [(c, lag_label(c, TRADE_LAG)) for c in V1_MARKET_COLS + V1_RENT_COLS if not c.startswith("전세피처출처")]
pairs += [(f"{c}_{s}", f"{c}_{'최근공표' if s == '전월' else s + '_최근공표'}") for c in ["주담대금리", "전세대출금리"]
          for s in ["전월", "3개월변화", "6개월변화"]]
corr = pd.DataFrame([{"V1": a, "V2": b, "상관": trade[f"{a}_v1"].astype(float).corr(trade[b].astype(float)),
                      "V1 결측": trade[f"{a}_v1"].isna().mean(), "V2 결측": trade[b].isna().mean()} for a, b in pairs])
display(corr.round(3))

[v2_trade] 행 234,302 | 열 279 | 모델 Feature 194 | 범주형 ['거래유형', '토지임대부아파트여부']
[v2_region_month] 행 1,500 | 열 140 | 모델 Feature 128 | 범주형 ['권역']
[v2_complex] 행 7,871 | 열 80 | 모델 Feature 52 | 범주형 []
V1 정책 열 이름이 남은 곳 (출처 확인): [['v2_trade', 'DSR단계', '03_policy'], ['v2_trade', '스트레스DSR단계', '03_policy'], ['v2_region_month', 'DSR단계', '03_policy'], ['v2_region_month', '스트레스DSR단계', '03_policy']]

v2_trade 결측이 있는 모델 Feature: 40


데이터구분,test,train,valid
승강기_100세대,0.4768,0.4938,0.4951
최근접중_학생1인당_체육장면적,0.1823,0.1922,0.2020
용적률,0.1852,0.1644,0.1663
건폐율,0.1838,0.1629,0.1657
세대당_주차,0.1629,0.1516,0.1505
최근접중_학생수증감률,0.0000,0.1444,0.0000
최근접초_학생수증감률,0.0029,0.1439,0.0006
예상가격_단지만,0.0447,0.1192,0.0484
거래유형,0.0000,0.0846,0.0000
상한구속_비중,0.0000,0.0543,0.0000


학습 구간 상수 — v2_trade: ['주담대_6억상한', '주담대_가격차등'] | v2_region_month: ['주담대_6억상한', '주담대_가격차등']


,V1,V2,상관,V1 결측,V2 결측
0,구_전월_중위가격,구_확정월_중위가격,0.944,0.020,0.00
1,구_전월_중위평당가,구_확정월_중위평당가,0.970,0.020,0.00
2,구_전월_거래량,구_확정월_거래량,0.679,0.020,0.00
3,구_과거3개월_중위가격,구_확정3개월_중위가격,0.992,0.049,0.00
4,구_과거6개월_중위가격,구_확정6개월_중위가격,0.997,0.069,0.00
5,구_과거3개월_거래량,구_확정3개월_거래량,0.933,0.049,0.00
6,구_과거6개월_거래량,구_확정6개월_거래량,0.968,0.069,0.00
7,구_전월_가격변화율,구_확정월_가격변화율,-0.244,0.037,0.00
8,서울_전월_중위가격,서울_확정월_중위가격,0.603,0.020,0.00
9,서울_과거3개월_중위가격,서울_확정3개월_중위가격,0.835,0.049,0.00


In [20]:
# ============================================================
# 단순 점검: 학습 구간에서 정답과의 순위 상관이 지나치게 높은 Feature (누수 의심 신호)
# - 같은 단지 과거 가격(예상가격)·공시가격은 정당하게 강한 Feature다. 0.99 넘는 것만 멈춘다.
# ============================================================
num_model = [c for c in trade_model if pd.api.types.is_numeric_dtype(trade[c]) and c not in const_trade]
sample = train.sample(50_000, random_state=0)
rho = sample[num_model].rank().corrwith(sample["log_평당가"].rank()).abs().sort_values(ascending=False)
display(rho.head(12).round(3).to_frame("|Spearman| vs log_평당가 (train 표본)"))
assert rho.max() < 0.99, rho.head(3)

region_num = [c for c in region_model if pd.api.types.is_numeric_dtype(region[c]) and c not in const_region]
lab = region_train.dropna(subset=["상승여부"])
rho_r = lab[region_num].rank().corrwith(lab["향후3개월_변화율"].rank()).abs().sort_values(ascending=False)
display(rho_r.head(10).round(3).to_frame("|Spearman| vs 향후3개월_변화율 (train)"))
assert rho_r.max() < 0.9, rho_r.head(3)

,|Spearman| vs log_평당가 (train 표본)
공시_m2당_면적대,0.934
공시_m2당_단지,0.912
공시기반_추정가,0.750
전세평당가_확정6개월,0.731
예상가격_단지만,0.722
전세평당가_확정3개월,0.720
전세평당가_확정월,0.709
구_확정월_중위평당가,0.702
구_확정3개월_중위가격,0.694
구_확정6개월_중위가격,0.694


,|Spearman| vs 향후3개월_변화율 (train)
공급_강도합_6m,0.245
구_확정월기준_평당가_6개월변화율,0.229
구_확정월기준_실현_3개월변화율,0.226
상한구속_비중,0.215
평균_한도비율,0.212
서울_순이동률_12m,0.203
구_확정3개월_전세가율,0.200
고가주택_주담대금지,0.194
전세대출금리_최근공표,0.194
분양공고세대_6m_서울,0.189


In [21]:
# ============================================================
# 저장
# ============================================================
for name, frame in [("v2_trade", trade), ("v2_region_month", region), ("v2_complex", cx)]:
    path = PROCESSED_DIR / f"{name}.parquet"
    frame.to_parquet(path, index=False)
    back = pd.read_parquet(path)
    assert back.shape == frame.shape
    print(f"저장: {path.name} {back.shape} ({path.stat().st_size / 1e6:.1f} MB)")
registry.to_csv(PROCESSED_DIR / "v2_feature_registry.csv", index=False, encoding="utf-8-sig")
print("저장: v2_feature_registry.csv", registry.shape)
display(registry.groupby(["dataset", "role"]).size().unstack(fill_value=0))

저장: v2_trade.parquet (234302, 279) (48.2 MB)
저장: v2_region_month.parquet (1500, 140) (0.7 MB)
저장: v2_complex.parquet (7871, 80) (2.5 MB)


저장: v2_feature_registry.csv (499, 8)


role,ablation,display,key,meta,model,target,v1_compare
dataset,,,,,,,
v2_complex,2,17,2,7,52,0,0
v2_region_month,6,0,1,3,128,2,0
v2_trade,15,6,3,12,194,4,45


## 11. 정리

- **시점 수정:** V1 시장·전세/월세·대출금리 피처를 공표·신고 시차에 맞춰 다시 만들었다 (V1 정의 그대로, 시차만 변경 → 재현 테스트로 확인).
  역·대규모점포·유치원 거리는 거래 시점에 운영 중이던 시설, 주요 일자리는 그 시점에 공표된 분기의 상위 10개 행정동 기준으로 바꿨다.
  역 출입구 거리를 새로 만들었다. 모두 V1 정의를 먼저 오차 0으로 재현한 뒤 시점만 바꿨다.
- **분류 정답:** V1과 같은 정의, 행 기준만 '예측시점 p월 1일'로 명확히 했다. V1 분류 피처는 기준월 당월 거래를 써서
  실제 예측 시점에는 알 수 없는 정보를 담았다 → 31에서 V1 피처 vs 확정월 피처 성능 차이를 보고한다.
- **11에서 할 비교:** `_v1` vs 확정월 피처, `전세피처출처_*`·`예상가격_출처` 포함/제외, 구 공급 원수치 vs 재고 비율,
  응급의료기관 · 유치원 거리(현재 목록, `ablation`) 포함/제외, 역 좌표 vs 역 출입구 거리.
- **11 1단계 결과 반영:** 예상가격 계열(`예상가격_만원` · `주담대_최대한도_만원` · `주담대_한도비율`)은 대체값 때문에
  신축 단지 오차를 키워 `ablation`으로 옮기고, 단지 자체 이력일 때만 값이 있는 `예상가격_단지만`을 기본 Feature로 쓴다.
- **한계:**
  - 실거래는 2026년에 받은 자료라 이후 해제된 거래가 빠져 있다 (당시에는 보였을 거래). 해제 비율 약 5%.
  - 2월이 낀 달은 신고 기한(30일)이 m월 1일보다 1~2일 늦게 끝날 수 있다 (확정월 마지막 1~2일 계약분).
  - 대출금리 원자료가 2020-12부터라 2021-07 행의 `_6개월변화_최근공표`는 결측이다.
  - 응급의료기관 거리는 현재 지정 목록 기준이다 (과거 목록 없음) → 11에서 포함/제외 Ablation.
  - 유치원은 개원일을 반영했지만, 원천이 현재 운영 목록이라 그 사이 폐원한 유치원은 빠져 있다 (과거 거리가 실제보다 길 수 있음).
  - 역 출입구 좌표는 현재 목록이다. 기존 역에 나중에 생긴 출입구는 구분하지 못한다 (새 역은 개통일 반영).